# LAB 2 — Advanced RAG: Ablação por Cenário em 2 Pipelines (A=Ollama / B=OpenSearch)
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Medir a **contribuição isolada** de cada técnica de Advanced RAG
(`rewriting`, `hybrid search`, `rerank`) em duas arquiteturas claramente separadas:

```
┌────────────────────────────────────────────────────────────────────────────┐
│                                                                            │
│   CAMINHO A — Python + Ollama (OpenSearch apenas para STORAGE)            │
│      LLM:        Groq llama-3.1-8b-instant (rewriting)                    │
│      Embeddings: Ollama bge-m3 (Python chama)                             │
│      Hybrid:     Python orquestra BM25 + kNN com vetor pré-calculado      │
│      Rerank:     Ollama qllama/bge-reranker-v2-m3 (Python chama)            │
│      OpenSearch: SÓ ARMAZENA vetores (zero modelo registrado)             │
│                                                                            │
│   CAMINHO B — OpenSearch ml-commons (servidor faz TUDO)                   │
│      Embeddings: text_embedding processor (all-MiniLM-L12-v2)             │
│      Hybrid:     normalization-processor (BM25 + neural query)             │
│      Rerank:     rerank-processor (ms-marco-MiniLM-L-12-v2)               │
│      Python:     APENAS dispara queries — zero compute local              │
│                                                                            │
│   ABLAÇÃO DE CENÁRIOS (em CADA caminho):                                  │
│      0. Baseline      = BM25 puro, sem rewriting, sem rerank              │
│      1. + Rewriting   = melhor variante de rewriting (BM25 puro)          │
│      2. + Hybrid      = BM25 + vetor (sem rewriting, sem rerank)          │
│      3. + Rerank      = BM25 + rerank (sem hybrid, sem rewriting)         │
│      4. + Hybrid+Rerank = BM25 + vetor + rerank (sem rewriting)           │
│      5. Advanced RAG completo = todas as 3 técnicas juntas                │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘
```

### Fluxo do lab

| Bloco | Passos | Conteúdo |
|---|---|---|
| **PARTE A** — Setup + matriz | 1–11 | Setup, embeddings Ollama, indexação (vetor pré-calc), hybrid Python, rerank Ollama, rewriting, matriz 5×5×2×2 = 100 buscas |
| **PARTE A** — Resultados | 12–14 | Tabela colorida, 6 cenários A com Δ vs baseline, barplot ablação |
| **PARTE B** — Setup + matriz | 15–21 | Plugin ml-commons, registro embed (all-MiniLM), ingest pipeline, índice neural, registro cross-encoder, search pipelines, função neural, matriz 5×5×2×2 = 100 buscas |
| **PARTE B** — Resultados | 22–24 | Tabela B, 6 cenários B com Δ vs baseline, barplot ablação |
| **COMPARAÇÃO A vs B** | 25–29 | Tabela 6 cenários × 2 caminhos, heatmap, barplot triplo, drill-down, análise de latência |
| **CONCLUSÃO** | 30 | Veredito + checklist + reflexão |

**Pré-requisitos:** Aula 1 (OpenSearch 3.x, Ollama, LangFuse) + modelos Ollama:
```
ollama pull bge-m3
ollama pull qllama/bge-reranker-v2-m3
```


---

## 🛠️ Passo 1 — Instalação e Imports

Tudo que o lab precisa em um único `pip install`. Como o lab é independente, instalamos:

- `opensearch-py` — cliente OpenSearch
- `langchain-ollama` + `langchain-huggingface` — embeddings (com fallback)
- `langchain-groq` + `openai` — LLM rewriter
- `transformers` + `torch` — fallback Python do reranker
- `langfuse>=2.36,<3` — observabilidade (mesma versão da Aula 1)
- `pandas`, `numpy`, `matplotlib` — análise e gráficos


In [1]:
# ════════════════════════════════════════════════════════════════════
# Instalação de dependências (idempotente)
# ════════════════════════════════════════════════════════════════════
%pip install -q opensearch-py langchain langchain-community langchain-openai \
    langchain-ollama langchain-huggingface langchain-groq groq openai \
    transformers torch sentence-transformers \
    "langfuse>=2.36,<3" python-dotenv \
    pandas numpy matplotlib
print("✅ Dependências instaladas")

Note: you may need to restart the kernel to use updated packages.
✅ Dependências instaladas


In [2]:
# ════════════════════════════════════════════════════════════════════
# Imports e configurações de ambiente
# ════════════════════════════════════════════════════════════════════
import os, json, time, uuid
import pandas as pd
import numpy as np
import requests
from typing import List, Dict, Optional
from dotenv import load_dotenv

# Carrega ~/mba-rag/.env (Aula 1) ou .env local
load_dotenv()

# ── Endpoints (Aula 1) ───────────────────────────────────────────────
OPENSEARCH_URL    = os.getenv("OPENSEARCH_URL", "http://localhost:9200")
OPENSEARCH_USER   = os.getenv("OPENSEARCH_USER", "admin")
OPENSEARCH_PASS   = os.getenv("OPENSEARCH_PASSWORD", "admin")
OLLAMA_BASE_URL   = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
LANGFUSE_HOST     = os.getenv("LANGFUSE_HOST", "http://localhost:3000")
LANGFUSE_PUBLIC   = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET   = os.getenv("LANGFUSE_SECRET_KEY", "")

# ── Modelos ──────────────────────────────────────────────────────────
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3:latest")
GROQ_API_KEY       = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL     = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")

# ── Configurações do lab ─────────────────────────────────────────────
INDEX_NAME       = os.getenv("INDEX_NAME", "corpus_juridico")
DATASET_PATH     = "../datasets/corpus_juridico_aula3.json"
EMBED_DIM        = 1024  # bge-m3
TOP_K_RETRIEVAL  = 10    # candidatos passados ao reranker (v6.1: era 20)
TOP_K_FINAL      = 5     # tamanho do resultado avaliado (Recall@5)

print("✅ Configurações carregadas:")
print(f"   OpenSearch: {OPENSEARCH_URL}")
print(f"   Ollama:     {OLLAMA_BASE_URL}  (embed: {OLLAMA_EMBED_MODEL})")
print(f"   LangFuse:   {LANGFUSE_HOST}  ({'keys OK' if LANGFUSE_PUBLIC and LANGFUSE_SECRET else 'SEM keys → observabilidade desligada'})")
print(f"   Groq:       {'API key OK' if GROQ_API_KEY else 'SEM key → fallback Ollama'} (modelo: {GROQ_LLM_MODEL})")
print(f"   Índice:     {INDEX_NAME} (será criado se não existir)")

✅ Configurações carregadas:
   OpenSearch: http://localhost:9200
   Ollama:     http://localhost:11434  (embed: bge-m3:latest)
   LangFuse:   http://localhost:3000  (keys OK)
   Groq:       API key OK (modelo: llama-3.1-8b-instant)
   Índice:     corpus_juridico (será criado se não existir)


---

## 🔌 Passo 2 — Conectar OpenSearch e LangFuse

Validamos que ambos os serviços da **Aula 1** estão rodando antes de prosseguir.
Se algum estiver fora, ativamos o modo *fallback* correspondente.


In [3]:
# ════════════════════════════════════════════════════════════════════
# Conexão OpenSearch + verificação do plugin ml-commons
# ════════════════════════════════════════════════════════════════════
from opensearchpy import OpenSearch

os_client = None
USE_OPENSEARCH = False
HAS_ML_COMMONS = False

try:
    os_client = OpenSearch(
        hosts=[OPENSEARCH_URL],
        http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
        use_ssl=OPENSEARCH_URL.startswith("https"),
        verify_certs=False,
        ssl_show_warn=False,
        timeout=30,
    )
    info = os_client.info()
    print(f"✅ OpenSearch {info['version']['number']} conectado em {OPENSEARCH_URL}")
    USE_OPENSEARCH = True

    # Verifica plugins instalados
    plugins = os_client.cat.plugins(format="json")
    plugin_names = {p["component"] for p in plugins}
    if "opensearch-ml" in plugin_names:
        HAS_ML_COMMONS = True
        print(f"✅ Plugin ml-commons disponível (rerank server-side ativo)")
    else:
        print(f"⚠️  Plugin ml-commons NÃO instalado — usaremos fallback Python para rerank")
        print(f"   Para habilitar: adicione `opensearch-ml` ao cluster")

except Exception as e:
    print(f"⚠️  OpenSearch indisponível ({e})")
    print(f"   O lab usará FAISS in-memory como fallback (limitado a kNN puro)")
    os_client = None
    USE_OPENSEARCH = False

✅ OpenSearch 3.0.0 conectado em http://localhost:9200
✅ Plugin ml-commons disponível (rerank server-side ativo)


In [4]:
# ════════════════════════════════════════════════════════════════════
# Conectar LangFuse (self-hosted, http://localhost:3000)
# ════════════════════════════════════════════════════════════════════
langfuse = None
LANGFUSE_ON = False

if LANGFUSE_PUBLIC and LANGFUSE_SECRET:
    try:
        from langfuse import Langfuse
        langfuse = Langfuse(
            public_key=LANGFUSE_PUBLIC,
            secret_key=LANGFUSE_SECRET,
            host=LANGFUSE_HOST,
        )
        # Smoke test: tenta criar um trace de aquecimento
        warmup = langfuse.trace(name="lab2-warmup", metadata={"phase": "smoke-test"})
        warmup.update(output={"ok": True})
        langfuse.flush()
        LANGFUSE_ON = True
        print(f"✅ LangFuse conectado em {LANGFUSE_HOST}")
        print(f"   SDK: langfuse-python ≥ 2.36")
        print(f"   UI:  abra {LANGFUSE_HOST} para inspecionar traces")
    except Exception as e:
        print(f"⚠️  LangFuse com problema ({e})")
        print(f"   Observabilidade DESLIGADA — métricas serão computadas apenas em memória")
        langfuse = None
else:
    print(f"ℹ️  LangFuse desativado (sem PUBLIC/SECRET no .env)")
    print(f"   Para ativar: defina LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY")

# Helper: cria um span seguro mesmo quando LangFuse está off
def lf_trace(name, **kwargs):
    """Retorna um trace LangFuse ou um stub no-op se desativado."""
    if LANGFUSE_ON and langfuse is not None:
        return langfuse.trace(name=name, **kwargs)
    return _NoopTrace()

class _NoopTrace:
    """Stub para quando LangFuse está desligado — preserva a API."""
    def span(self, **kw):  return self
    def update(self, **kw): return self
    def score(self, **kw):  return self
    def end(self, **kw):    return self
    def __enter__(self):    return self
    def __exit__(self, *a): return False

print(f"✅ Helper `lf_trace()` pronto (modo: {'LIVE' if LANGFUSE_ON else 'NO-OP'})")

✅ LangFuse conectado em http://localhost:3000
   SDK: langfuse-python ≥ 2.36
   UI:  abra http://localhost:3000 para inspecionar traces
✅ Helper `lf_trace()` pronto (modo: LIVE)


---

## 🧮 Passo 3 — Embeddings: Cliente Ollama + Fallback HuggingFace

Para indexar e buscar, precisamos vetorizar texto. Vamos usar **BGE-M3** (1024 dims,
multilíngue) — o mesmo modelo do LAB1 — servido pelo **Ollama** local (`bge-m3`).

Se o Ollama estiver fora ou o modelo não estiver baixado, caímos automaticamente para
`HuggingFaceEmbeddings(BAAI/bge-m3)` (mesmo espaço vetorial, executa no Python).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Inicialização dos embeddings com fallback automático
# ════════════════════════════════════════════════════════════════════
embeddings = None
EMBED_BACKEND = None

try:
    from langchain_ollama import OllamaEmbeddings
    candidate = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    _ = candidate.embed_query("teste de conexão")
    embeddings = candidate
    EMBED_BACKEND = f"Ollama ({OLLAMA_EMBED_MODEL})"
    print(f"✅ Embeddings: {EMBED_BACKEND}")
except Exception as e:
    print(f"⚠️  Ollama embeddings indisponível ({e}); tentando HuggingFace…")

if embeddings is None:
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    EMBED_BACKEND = "HuggingFace (BAAI/bge-m3)"
    print(f"✅ Embeddings (fallback): {EMBED_BACKEND}")

print(f"   Dimensão: {EMBED_DIM} | espaço vetorial: cosine")

---

## 📚 Passo 4 — Carregar o Corpus Jurídico e o Ground Truth

Reusamos o mesmo `corpus_juridico_aula3.json` do LAB1 (80 docs + 10 queries com
`documentos_relevantes`). Isso garante **comparabilidade direta** dos resultados.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Carregar corpus e construir ground truth
# ════════════════════════════════════════════════════════════════════
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

documentos       = dataset["documentos"]
queries_baseline = dataset["queries_baseline"][:5]  # 5 primeiras (mesmas do LAB1)
ground_truth     = {q["id"]: set(q.get("documentos_relevantes", [])) for q in dataset["queries_baseline"]}

print(f"✅ Corpus carregado: {len(documentos)} docs, {len(queries_baseline)} queries de teste")
print()
print(f"📋 Queries do teste e tamanho do ground truth:")
for q in queries_baseline:
    rels = ground_truth.get(q["id"], set())
    print(f"   {q['id']}: {len(rels):>2} rel.  →  {q['query_original'][:55]}")

---

## 📥 Passo 5A — Indexação Client-Side (Embeddings via Ollama)

**Estratégia A**: geramos os embeddings no Python (Ollama bge-m3) e enviamos ao
OpenSearch via `bulk` já com o vetor pronto. Foi a estratégia usada no LAB1.

| ✅ Vantagens                          | ⚠️ Desvantagens                       |
|---------------------------------------|---------------------------------------|
| Independe de ml-commons               | Embedding viaja pela rede             |
| Modelo fica isolado do cluster        | Reindexação exige rodar Python        |
| Fácil de testar e debugar             | Sem invocação automática em buscas    |


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Indexação client-side: embeddings gerados em Python, enviados ao OpenSearch
# ════════════════════════════════════════════════════════════════════
from opensearchpy.helpers import bulk

INDEX_MAPPING = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 100,
            "default_pipeline": None,  # sem ingest pipeline (caminho A)
        }
    },
    "mappings": {
        "properties": {
            "texto":          {"type": "text", "analyzer": "portuguese"},
            "ementa":         {"type": "text", "analyzer": "portuguese"},
            "palavras_chave": {"type": "keyword"},
            "tipo":           {"type": "keyword"},
            "tribunal":       {"type": "keyword"},
            "numero":         {"type": "keyword"},
            "data":           {"type": "date", "ignore_malformed": True},
            "embedding": {
                "type": "knn_vector",
                "dimension": EMBED_DIM,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {"ef_construction": 128, "m": 16},
                },
            },
        }
    },
}

if USE_OPENSEARCH:
    try:
        if os_client.indices.exists(index=INDEX_NAME):
            print(f"ℹ️  Índice '{INDEX_NAME}' existe — recriando para garantir mapping consistente com o LAB2 v3")
            os_client.indices.delete(index=INDEX_NAME)
        os_client.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
        print(f"✅ Índice '{INDEX_NAME}' criado")

        # Pré-computa embeddings
        textos = [f"{d.get('ementa','')}\\n\\n{d.get('texto','')}".strip() for d in documentos]
        print(f"⏳ Gerando {len(textos)} embeddings com {EMBED_BACKEND}…")
        t0 = time.time()
        vetores = embeddings.embed_documents(textos)
        print(f"✅ Embeddings gerados em {time.time()-t0:.1f}s")

        # Bulk
        actions = [
            {
                "_index": INDEX_NAME,
                "_id":    d["id"],
                "_source": {
                    "texto":          d.get("texto", ""),
                    "ementa":         d.get("ementa", ""),
                    "palavras_chave": d.get("palavras_chave", []),
                    "tipo":           d.get("tipo", ""),
                    "tribunal":       d.get("tribunal", ""),
                    "numero":         d.get("numero", d["id"]),
                    "data":           d.get("data", ""),
                    "embedding":      v,
                },
            }
            for d, v in zip(documentos, vetores)
        ]
        success, errors = bulk(os_client, actions, refresh="wait_for")
        print(f"✅ Bulk: {success} docs indexados, erros={len(errors) if isinstance(errors, list) else errors}")
        count = os_client.count(index=INDEX_NAME)["count"]
        print(f"   Total no índice: {count}")
    except Exception as e:
        print(f"❌ Falha na indexação: {e}")
        USE_OPENSEARCH = False
else:
    print("⚠️  OpenSearch off — pulando indexação")

---

## 🔀 Passo 6 — Hybrid Search com `normalization-processor`

Hybrid Search no OpenSearch usa um **search pipeline** com `normalization-processor`
que combina os scores de duas (ou mais) cláusulas — tipicamente BM25 (`match`) e
kNN/Neural (`knn`) — após normalização por `min_max` e agregação por `arith_mean`.

📖 Doc: https://docs.opensearch.org/latest/vector-search/ai-search/hybrid-search/index/

```
Query  ──┬─►  match (BM25)       ┐
         │                       ├──► normalization (min_max)
         └─►  knn (cosine bge-m3)┘             │
                                               ▼
                                     arith_mean (0.3 / 0.7)
                                               │
                                               ▼
                                       top-N ordenado
```


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Criar search pipeline hybrid (BM25 + kNN com normalization)
# ════════════════════════════════════════════════════════════════════
HYBRID_PIPELINE = "lab2-hybrid-pipeline"

hybrid_pipeline_body = {
    "description": "Hybrid search: BM25 + kNN com min_max normalization e arith_mean",
    "phase_results_processors": [
        {
            "normalization-processor": {
                "normalization": {"technique": "min_max"},
                "combination": {
                    "technique": "arithmetic_mean",
                    "parameters": {"weights": [0.3, 0.7]}  # 30% BM25, 70% vetor
                }
            }
        }
    ]
}

if USE_OPENSEARCH:
    try:
        os_client.transport.perform_request(
            "PUT", f"/_search/pipeline/{HYBRID_PIPELINE}",
            body=hybrid_pipeline_body
        )
        print(f"✅ Search pipeline '{HYBRID_PIPELINE}' criado")
        print(f"   Combinação: 30% BM25 + 70% kNN (min_max → arith_mean)")
    except Exception as e:
        print(f"⚠️  Falha ao criar pipeline hybrid: {e}")
        print(f"   Buscas usarão script_score como alternativa")
else:
    print("ℹ️  Sem OpenSearch — hybrid pipeline pulado")

---

## 🦙 Passo 7 — Caminho A: "Reranker" via Ollama (`qllama/bge-reranker-v2-m3`)

### ⚠️ Nota didática importante: pseudo-rerank vs cross-encoder real

O modelo `qllama/bge-reranker-v2-m3` disponível no Ollama é uma **quantização** do
[`BAAI/bge-reranker-v2-m3`](https://huggingface.co/BAAI/bge-reranker-v2-m3) original.
Ao ser servido pelo Ollama, ele **perde a cabeça de classificação** que produziria o
escalar de relevância e passa a operar como **bi-encoder** (devolve vetor de 1024 floats).

| Aspecto | Cross-encoder real (HuggingFace) | **Pseudo-rerank via Ollama (este lab)** |
|---|---|---|
| API | `model([q, d]) → 1 escalar` | `/api/embed(q) → vetor 1024d` |
| Como funciona | Processa par `(q, d)` JUNTO | Encoda q e d SEPARADAMENTE, depois cosseno |
| Qualidade típica | ~85-90% Recall@5 | ~70-75% Recall@5 |
| Latência | 50-200ms/par | 30-50ms/embed × (1+N) |
| Custo | Caro (transformer pesado) | Barato (bi-encoder leve) |

### Por que mantemos assim?

1. ✅ É **o caminho oficial documentado pela Ollama** (ver doc do modelo)
2. ✅ Mantém **pureza arquitetural** do caminho A (tudo via Ollama)
3. ✅ **Honestidade pedagógica** — aluno vê o que dá pra fazer hoje no Ollama
4. ✅ Permite comparação justa com **caminho B** (cross-encoder real via ml-commons)

### Implementação seguindo a [doc oficial](https://ollama.com/qllama/bge-reranker-v2-m3)

```python
# 1. Embed query
q_emb = ollama.embed(model="qllama/bge-reranker-v2-m3", input=query)

# 2. Embed N docs (em paralelo)
d_embs = [ollama.embed(...) for doc in docs]

# 3. Cosseno query × doc, ordena, retorna top-K
scores = [cosine(q_emb, d_emb) for d_emb in d_embs]
```

Esta célula apenas **verifica/instala** o modelo no Ollama.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Verifica/instala qllama/bge-reranker-v2-m3 no Ollama
# ════════════════════════════════════════════════════════════════════
# Mudança v6.2: usar o registro qllama/ (mais downloads, recomendado pela Ollama)
RERANKER_OLLAMA_MODEL = os.getenv("RERANKER_OLLAMA_MODEL", "qllama/bge-reranker-v2-m3")
RERANK_OLLAMA_OK = False

print(f"🔍 Verificando '{RERANKER_OLLAMA_MODEL}' no Ollama…")
try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    r.raise_for_status()
    modelos = [m["name"] for m in r.json().get("models", [])]
    base_names = {m.split(":")[0] for m in modelos}
    target_base = RERANKER_OLLAMA_MODEL.split(":")[0]

    if RERANKER_OLLAMA_MODEL in modelos or target_base in base_names:
        print(f"✅ Reranker '{RERANKER_OLLAMA_MODEL}' já disponível")
        RERANK_OLLAMA_OK = True
    else:
        print(f"⏳ Não encontrado — fazendo pull (~636 MB)…")
        with requests.post(
            f"{OLLAMA_BASE_URL}/api/pull",
            json={"name": RERANKER_OLLAMA_MODEL, "stream": False},
            timeout=600
        ) as resp:
            if resp.status_code == 200:
                print(f"✅ Pull concluído: {RERANKER_OLLAMA_MODEL}")
                RERANK_OLLAMA_OK = True
            else:
                print(f"⚠️  Pull falhou (HTTP {resp.status_code})")
                print(f"   Execute manualmente: ollama pull {RERANKER_OLLAMA_MODEL}")
except Exception as e:
    print(f"❌ Ollama indisponível: {e}")
    print(f"   Caminho A funcionará SEM rerank (só hybrid retrieval).")

if RERANK_OLLAMA_OK:
    print(f"\n📊 Modelo pronto para uso como bi-encoder:")
    print(f"   Nome:       {RERANKER_OLLAMA_MODEL}")
    print(f"   Tag:        embedding (no Ollama registry)")
    print(f"   Endpoint:   /api/embed (ou /api/embeddings deprecated)")
    print(f"   Estratégia: cosseno entre embedding da query e dos docs")
else:
    print(f"\n⚠️  Rerank no caminho A: DESABILITADO")

---

## 🎓 Passo 7.5 — Discussão Didática: Limitação do Rerank no Ollama e 3 Estratégias

> 📚 **Esta é uma das seções mais importantes pedagogicamente do lab.** Pare aqui
> e leia com atenção antes de prosseguir.

### 🐛 O problema técnico

O modelo `BAAI/bge-reranker-v2-m3` original do Hugging Face é um **cross-encoder** —
ele recebe um par `(query, documento)` e produz **um único escalar** de relevância
através de uma "cabeça de classificação" (classification head) que segue o transformer.

```
[CLS] query [SEP] documento [SEP]
       │
       ▼
   Transformer (BERT-like)
       │
       ▼
   [CLS] token embedding
       │
       ▼
   Linear layer ← ESTA É A CABEÇA DE CLASSIFICAÇÃO
       │
       ▼
   Sigmoid → score ∈ [0, 1]
```

**O problema:** quando este modelo é **quantizado para GGUF** (formato do Ollama),
a cabeça de classificação **é descartada**. O que sobra responde via `/api/embed`
devolvendo **um vetor de 1024 floats** (o embedding do `[CLS]` token), funcionando
como **bi-encoder** em vez de cross-encoder.

Isso significa que `qllama/bge-reranker-v2-m3` no Ollama **não pode produzir o
escalar de relevância** que o modelo foi treinado para gerar.

### 💡 As 3 estratégias para contornar (escolhíveis no Passo 8)

Configure em `RERANK_STRATEGY` qual usar (ou no `.env`):

```python
RERANK_STRATEGY = "transformers_ce"   # padrão recomendado
# alternativas: "ollama_cosine" | "flashrank_onnx"
```

#### A) `ollama_cosine` — Pseudo-rerank via cosseno (workaround Ollama)

```
1. /api/embed(query)              → vetor 1024d
2. /api/embed(doc) × N (paralelo) → N vetores 1024d
3. cosine(q, doc) → score por par
4. Ordena descendente → top_k
```

- ✅ **Pureza Ollama**: tudo passa pelo daemon do Ollama
- ✅ **Sem dependências extras**
- ❌ **Não é cross-encoder de verdade** — usa bi-encoder degradado
- ❌ **Qualidade limitada** (~70% Recall@5 típico)
- ❌ **Pode regredir o ranking** se o bi-encoder do reranker for pior que o bge-m3
   do retrieval (foi exatamente o que vimos: 0.317 → 0.100)

#### B) `transformers_ce` — Cross-encoder REAL via Hugging Face *(recomendado)*

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-v2-m3")
score = sigmoid(model(query, doc).logits)
```

- ✅ **Cross-encoder de verdade** — processa par junto, captura interações query-doc
- ✅ **Qualidade alta** (~88% Recall@5 típico)
- ✅ **Multilíngue forte** (PT-BR jurídico funciona muito bem)
- ⚠️ **Carrega torch + transformers** (~2 GB de dependências)
- ⚠️ **Modelo ~560 MB** baixado do HuggingFace na primeira vez
- ❌ **Quebra a "pureza Ollama"** do caminho A

#### C) `flashrank_onnx` — Cross-encoder ONNX leve

```python
from flashrank import Ranker, RerankRequest
ranker = Ranker(model_name="ms-marco-MultiBERT-L-12")  # 150 MB, 100+ idiomas
results = ranker.rerank(RerankRequest(query=q, passages=[...]))
```

- ✅ **Cross-encoder real** com quantização INT8 ONNX
- ✅ **Muito rápido em CPU** (2-5× mais rápido que transformers vanilla)
- ✅ **Pequeno** (150 MB para o modelo multilíngue)
- ✅ **Sem torch** (usa onnxruntime)
- ⚠️ **Não tem BGE-Reranker no catálogo** (alternativa: ms-marco-MultiBERT-L-12, qualidade menor que BGE)
- ⚠️ **Qualidade intermediária** (~82% Recall@5 esperado)

### 📊 Tabela comparativa final

| Estratégia | Tipo | Qualidade | Latência | Tamanho | Dependências |
|---|---|---|---|---|---|
| `ollama_cosine` (v6.2) | bi-encoder + cosseno | ⭐⭐ (~70%) | ~30ms/par | 636 MB | só Ollama |
| `transformers_ce` ⭐ padrão | cross-encoder real | ⭐⭐⭐⭐⭐ (~88%) | ~80ms/par | ~560 MB + torch | torch + transformers |
| `flashrank_onnx` | cross-encoder ONNX | ⭐⭐⭐⭐ (~82%) | ~20ms/par | 150 MB | onnxruntime |

### 🎯 Recomendação de uso

- **Para o lab/MBA**: use `transformers_ce` (padrão) para ter as melhores métricas
- **Para produção CPU-only com restrição de RAM**: use `flashrank_onnx`
- **Para demonstrar limitação do Ollama**: rode `ollama_cosine` e compare com B no Passo 11.5
- **Em produção real com GPU**: `transformers_ce` em GPU (10× mais rápido que CPU)

> 💡 **Importante**: a "pureza Ollama" não é um valor em si — é uma escolha
> arquitetural. Para **qualidade de RAG jurídico**, recomendamos abrir mão dela
> e usar o cross-encoder real. O caminho B (server-side OpenSearch) continua
> "puro" do lado servidor.

### 🔬 Como vamos demonstrar

- **Passo 8**: implementa as 3 estratégias num único dispatcher `rerank_caminho_a()`
- **Passo 10.5**: sanity check roda as 3 estratégias com 1 par (rel, irr) e mostra discriminação lado a lado
- **Passo 11.5**: mini-comparativo execução cenário 5 × 3 estratégias × 5 queries (~15 buscas extras)


---

## 🔄 Passo 8 — Caminho A: Função `rerank_ollama()` via Cosseno

Implementação alinhada com a [doc oficial do `qllama/bge-reranker-v2-m3`](https://ollama.com/qllama/bge-reranker-v2-m3):

```
1. /api/embed(query)         → 1 vetor 1024d (1 chamada)
2. /api/embed(doc) × N       → N vetores 1024d (paralelo, 8 workers)
3. cosseno(q, d) para cada par → N scores
4. Ordena descendente, retorna top_k
```

### Decisões de implementação

- **`/api/embed`** (endpoint novo) primário, com fallback `/api/embeddings` (deprecated)
- **Cache da query**: 1 chamada para `query`, reusada para todos os docs
- **Normalização L2** dos vetores → cosseno = dot product (mais rápido)
- **ThreadPoolExecutor(8 workers)** para os N embeddings dos docs (C1 do v6.1)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 8 — Dispatcher de 3 estratégias de rerank no caminho A
# ════════════════════════════════════════════════════════════════════
import numpy as np
from concurrent.futures import ThreadPoolExecutor

RERANK_STRATEGY = os.getenv("RERANK_STRATEGY", "transformers_ce")
# alternativas: "ollama_cosine" | "transformers_ce" | "flashrank_onnx"

RERANK_MAX_WORKERS = int(os.getenv("RERANK_MAX_WORKERS", "8"))

# Cache de modelos carregados (lazy loading)
_ce_tokenizer = None
_ce_model     = None
_flashrank_ranker = None

# ─────────────────────────────────────────────────────────────────────
# Estratégia A: ollama_cosine (pseudo-rerank via bi-encoder + cosseno)
# ─────────────────────────────────────────────────────────────────────

def _ollama_embed(text: str):
    """Tenta /api/embed (novo); fallback para /api/embeddings (deprecated)."""
    try:
        r = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": RERANKER_OLLAMA_MODEL, "input": text},
            timeout=30,
        )
        if r.status_code == 200:
            embs = r.json().get("embeddings")
            if embs and isinstance(embs, list) and embs[0]:
                return np.array(embs[0], dtype=np.float32)
    except Exception:
        pass
    try:
        r = requests.post(
            f"{OLLAMA_BASE_URL}/api/embeddings",
            json={"model": RERANKER_OLLAMA_MODEL, "prompt": text},
            timeout=30,
        )
        if r.status_code == 200:
            emb = r.json().get("embedding")
            if emb and isinstance(emb, list):
                return np.array(emb, dtype=np.float32)
    except Exception:
        pass
    return None


def _normalize(v):
    n = np.linalg.norm(v)
    return v / (n + 1e-12)


def rerank_ollama_cosine(query: str, docs: list, top_k: int = 5) -> list:
    """Estratégia A: pseudo-rerank cosseno via Ollama (bi-encoder)."""
    if not docs or not RERANK_OLLAMA_OK:
        return docs[:top_k]

    def doc_text(d):
        return d.get("texto") or d.get("ementa") or ""

    q_emb = _ollama_embed(query)
    if q_emb is None:
        return docs[:top_k]
    q_norm = _normalize(q_emb)

    with ThreadPoolExecutor(max_workers=RERANK_MAX_WORKERS) as ex:
        d_embs = list(ex.map(lambda d: _ollama_embed(doc_text(d)), docs))

    scores = []
    for d_emb in d_embs:
        if d_emb is None:
            scores.append(0.0)
        else:
            scores.append(float(np.dot(q_norm, _normalize(d_emb))))

    reranked = [{**d, "score_reranker": round(s, 4)}
                for d, s in zip(docs, scores)]
    return sorted(reranked, key=lambda x: x["score_reranker"], reverse=True)[:top_k]


# ─────────────────────────────────────────────────────────────────────
# Estratégia B: transformers_ce (cross-encoder REAL via HF)
# ─────────────────────────────────────────────────────────────────────

def _load_ce_model():
    """Lazy load do BGE-Reranker via transformers."""
    global _ce_tokenizer, _ce_model
    if _ce_tokenizer is not None:
        return True
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        print("⏳ Carregando BAAI/bge-reranker-v2-m3 (cross-encoder, ~560 MB)…")
        _ce_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-v2-m3")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype  = torch.float16 if device == "cuda" else torch.float32
        _ce_model = AutoModelForSequenceClassification.from_pretrained(
            "BAAI/bge-reranker-v2-m3", torch_dtype=dtype
        ).eval().to(device)
        print(f"✅ Cross-encoder carregado em {device.upper()}")
        return True
    except Exception as e:
        print(f"❌ Falha ao carregar cross-encoder: {e}")
        return False


def rerank_transformers_ce(query: str, docs: list, top_k: int = 5) -> list:
    """Estratégia B: cross-encoder REAL via HuggingFace transformers."""
    if not docs:
        return []
    if not _load_ce_model():
        return docs[:top_k]

    import torch

    def doc_text(d):
        return d.get("texto") or d.get("ementa") or ""

    pairs = [[query, doc_text(d)] for d in docs]
    device = next(_ce_model.parameters()).device

    scores = []
    BATCH = 8
    for i in range(0, len(pairs), BATCH):
        batch = pairs[i:i+BATCH]
        inputs = _ce_tokenizer(batch, padding=True, truncation=True,
                                max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = _ce_model(**inputs).logits.squeeze(-1)
            batch_scores = torch.sigmoid(logits).cpu().numpy()
        scores.extend(batch_scores.tolist() if batch_scores.ndim > 0 else [float(batch_scores)])

    reranked = [{**d, "score_reranker": round(float(s), 4)}
                for d, s in zip(docs, scores)]
    return sorted(reranked, key=lambda x: x["score_reranker"], reverse=True)[:top_k]


# ─────────────────────────────────────────────────────────────────────
# Estratégia C: flashrank_onnx (cross-encoder leve via ONNX)
# ─────────────────────────────────────────────────────────────────────

def _load_flashrank():
    """Lazy load do flashrank com modelo multilíngue."""
    global _flashrank_ranker
    if _flashrank_ranker is not None:
        return True
    try:
        # Install lazy
        try:
            from flashrank import Ranker, RerankRequest
        except ImportError:
            print("⏳ Instalando flashrank…")
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flashrank"])
            from flashrank import Ranker, RerankRequest

        print("⏳ Carregando flashrank ms-marco-MultiBERT-L-12 (~150 MB, multilíngue)…")
        _flashrank_ranker = Ranker(model_name="ms-marco-MultiBERT-L-12", max_length=512)
        print(f"✅ FlashRank carregado")
        return True
    except Exception as e:
        print(f"❌ Falha ao carregar flashrank: {e}")
        return False


def rerank_flashrank(query: str, docs: list, top_k: int = 5) -> list:
    """Estratégia C: cross-encoder ONNX via flashrank (CPU-friendly)."""
    if not docs:
        return []
    if not _load_flashrank():
        return docs[:top_k]

    from flashrank import RerankRequest

    def doc_text(d):
        return d.get("texto") or d.get("ementa") or ""

    passages = [
        {"id": str(i), "text": doc_text(d), "meta": {"orig_idx": i}}
        for i, d in enumerate(docs)
    ]
    req = RerankRequest(query=query, passages=passages)
    results = _flashrank_ranker.rerank(req)

    # Reconstrói a lista preservando metadados originais
    id2score = {r["id"]: r["score"] for r in results}
    reranked = []
    for i, d in enumerate(docs):
        s = id2score.get(str(i), 0.0)
        reranked.append({**d, "score_reranker": round(float(s), 4)})
    return sorted(reranked, key=lambda x: x["score_reranker"], reverse=True)[:top_k]



# ─────────────────────────────────────────────────────────────────────
# Helper: verifica se a estratégia ativa está REALMENTE disponível
# Usado pelo Passo 11 para decidir se inclui "on" em rerank_modes_a
# ─────────────────────────────────────────────────────────────────────

def _strategy_disponivel(strategy: str = None) -> tuple[bool, str]:
    """Verifica se a estratégia de rerank está pronta para uso.

    Returns:
        (disponivel: bool, mensagem: str) — mensagem com causa em caso de falha.
    """
    s = strategy or RERANK_STRATEGY

    if s == "ollama_cosine":
        if RERANK_OLLAMA_OK:
            return True, f"Ollama OK em {OLLAMA_BASE_URL}"
        return False, "Ollama não está rodando ou modelo qllama/bge-reranker-v2-m3 indisponível"

    elif s == "transformers_ce":
        # Tenta carregar (lazy) — se falhar, _load_ce_model imprime erro e retorna False
        ok = _load_ce_model()
        if ok:
            return True, "Cross-encoder BAAI/bge-reranker-v2-m3 carregado via HF transformers"
        return False, "Falha ao carregar BAAI/bge-reranker-v2-m3 (sem internet, sem disco, ou erro de torch?)"

    elif s == "flashrank_onnx":
        ok = _load_flashrank()
        if ok:
            return True, "FlashRank ms-marco-MultiBERT-L-12 carregado (ONNX)"
        return False, "Falha ao carregar flashrank (pip install falhou ou modelo indisponível?)"

    return False, f"Estratégia desconhecida: {s}"


# ─────────────────────────────────────────────────────────────────────
# Dispatcher principal
# ─────────────────────────────────────────────────────────────────────

def rerank_caminho_a(query: str, docs: list, top_k: int = 5,
                      strategy: str = None) -> list:
    """Dispatcher para as 3 estratégias de rerank do caminho A.

    Args:
        strategy: força uma estratégia específica (None = usa RERANK_STRATEGY global)
    """
    s = strategy or RERANK_STRATEGY
    if s == "ollama_cosine":
        return rerank_ollama_cosine(query, docs, top_k)
    elif s == "transformers_ce":
        return rerank_transformers_ce(query, docs, top_k)
    elif s == "flashrank_onnx":
        return rerank_flashrank(query, docs, top_k)
    else:
        raise ValueError(f"RERANK_STRATEGY desconhecida: {s}. "
                          "Use: ollama_cosine | transformers_ce | flashrank_onnx")


# Alias retrocompatibilidade: chamadas a rerank_ollama() vão para o dispatcher
def rerank_ollama(query: str, docs: list, top_k: int = 5) -> list:
    """Alias retrocompatível — usa a estratégia ativa em RERANK_STRATEGY."""
    return rerank_caminho_a(query, docs, top_k)


print(f"✅ 3 estratégias de rerank registradas:")
print(f"   • ollama_cosine    (pseudo-rerank Ollama via cosseno)")
print(f"   • transformers_ce  (cross-encoder REAL via HuggingFace) ← padrão")
print(f"   • flashrank_onnx   (cross-encoder ONNX leve)")
print()
print(f"📌 RERANK_STRATEGY ativa: '{RERANK_STRATEGY}'")
print(f"   Mude via .env (RERANK_STRATEGY=...) ou no notebook antes do Passo 11")
print(f"   Os modelos são carregados sob demanda na 1ª chamada (lazy loading)")

---

## 🔍 Passo 9 — Caminho A: Função `buscar_corpus()` com 4 Combinações

A função recebe 2 parâmetros independentes que controlam as 3 técnicas:

| Parâmetro | Valores | Significado |
|---|---|---|
| `retrieval_mode` | `"bm25"` | BM25 puro (`multi_match`) — linha base |
| | `"hybrid"` | BM25 + kNN com vetor calculado em Python (hybrid pipeline) |
| `rerank_mode` | `"off"` | Sem rerank |
| | `"ollama"` | Aplica `rerank_ollama` (cross-encoder Ollama) |

Combinado com 5 variantes de rewriting (Passo 10), gera a matriz 5×5×2×2 = 100 buscas.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# buscar_corpus() — caminho A com 4 combinações retrieval × rerank
# ════════════════════════════════════════════════════════════════════

RERANK_OLLAMA_OK = False

def _build_bm25_query(query: str, size: int) -> dict:
    """BM25 puro: multi_match sem vetor — linha base do caminho A."""
    return {
        "size": size,
        "query": {
            "multi_match": {
                "query":  query,
                "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                "type":   "best_fields",
            }
        }
    }


def _build_hybrid_query_clientside(query: str, q_vec: list, size: int) -> dict:
    """Hybrid no caminho A: BM25 + kNN com vetor JÁ calculado em Python."""
    return {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "multi_match": {
                            "query":  query,
                            "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                            "type":   "best_fields",
                        }
                    },
                    {
                        "knn": {
                            "embedding": {
                                "vector": q_vec,
                                "k":      size,
                            }
                        }
                    }
                ]
            }
        }
    }


def buscar_corpus(query: str, top_k: int = 5,
                  retrieval_mode: str = "bm25",
                  rerank_mode: str = "off") -> list:
    """Busca no índice corpus_juridico (caminho A — Ollama + OpenSearch storage).

    Args:
        query           : texto da pergunta (já reescrito se for o caso)
        top_k           : nº de docs finais
        retrieval_mode  : "bm25" | "hybrid"
        rerank_mode     : "off" | "ollama"

    Returns:
        Lista de dicts {id, score, fonte, texto, ementa, [score_reranker]}
    """
    if not USE_OPENSEARCH:
        return []

    # Decide tamanho do candidato (se rerank ligado, pega top-20)
    candidate_size = TOP_K_RETRIEVAL if rerank_mode != "off" else top_k

    # === Etapa 1: retrieval ===
    if retrieval_mode == "bm25":
        body = _build_bm25_query(query, candidate_size)
        try:
            resp = os_client.search(index=INDEX_NAME, body=body)
        except Exception as e:
            print(f"⚠️  BM25 search falhou: {e}")
            return []
    elif retrieval_mode == "hybrid":
        # Embedding gerado em Python via Ollama
        q_vec = embeddings.embed_query(query)
        body = _build_hybrid_query_clientside(query, q_vec, candidate_size)
        try:
            resp = os_client.transport.perform_request(
                "POST", f"/{INDEX_NAME}/_search?search_pipeline={HYBRID_PIPELINE}",
                body=body
            )
        except Exception as e:
            print(f"⚠️  Hybrid search falhou ({e}); caindo para BM25")
            try:
                resp = os_client.search(index=INDEX_NAME, body=_build_bm25_query(query, candidate_size))
            except Exception:
                return []
    else:
        raise ValueError(f"retrieval_mode desconhecido: {retrieval_mode}")

    candidatos = [_hit_to_dict(h) for h in resp["hits"]["hits"]]

    # === Etapa 2: rerank ===
    if rerank_mode == "off":
        return candidatos[:top_k]
    elif rerank_mode == "on":
        return rerank_ollama(query, candidatos, top_k=top_k)
    else:
        raise ValueError(f"rerank_mode desconhecido: {rerank_mode}")


def _hit_to_dict(h) -> dict:
    return {
        "id":     h["_id"],
        "score":  round(h["_score"], 4),
        "fonte":  h["_source"].get("numero", h["_id"]),
        "texto":  h["_source"].get("texto", ""),
        "ementa": h["_source"].get("ementa", ""),
    }


print(f"✅ Função buscar_corpus() pronta (caminho A — Ollama puro)")
print(f"   retrieval_mode: bm25 | hybrid")
print(f"   rerank_mode:    off | on  ({'ON' if RERANK_OLLAMA_OK else 'OFF'})")
print(f"   Combinações:    4 (testaremos todas na matriz)")

---

## 🤖 Passo 10 — Gerar Variantes de Query (rewriting)

Para comparar com o LAB1, recriamos as **4 variantes de rewriting** + a original:
`paraphrase_1`, `paraphrase_2`, `hyde_lite`, `step_back`. LLM: Groq
`llama-3.1-8b-instant` com fallback Ollama.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Setup LLM: Groq primário + Ollama fallback (mesmo padrão do LAB1)
# ════════════════════════════════════════════════════════════════════
from openai import OpenAI

llm_client = None
MODEL_NAME = None
LLM_PROVIDER = None

if GROQ_API_KEY:
    try:
        c = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        llm_client, MODEL_NAME, LLM_PROVIDER = c, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM: Groq ({MODEL_NAME})")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); usando Ollama…")

if llm_client is None:
    try:
        c = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        OLLAMA_LLM = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")
        c.chat.completions.create(model=OLLAMA_LLM,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        llm_client, MODEL_NAME, LLM_PROVIDER = c, OLLAMA_LLM, "ollama"
        print(f"✅ LLM: Ollama ({MODEL_NAME})")
    except Exception as e:
        print(f"❌ Nenhum LLM disponível: {e}")


def llm_call(prompt: str, temperature: float = 0.3, max_tokens: int = 400) -> str:
    if llm_client is None: return ""
    try:
        r = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role":"user","content":prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return ""


def gerar_variantes(query_original: str) -> Dict[str, str]:
    """Gera 4 variantes de rewriting + retorna a original."""
    # Paráfrases (2 técnicas)
    p_prompt = f"""Reformule esta pergunta jurídica em 2 variações usando terminologia técnica.
Pergunta: {query_original}
Retorne SÓ as 2 variações, uma por linha, sem numeração."""
    paras = llm_call(p_prompt, temperature=0.5).split("\\n")
    paras = [p.strip() for p in paras if p.strip()][:2]
    while len(paras) < 2:
        paras.append(query_original)

    # HyDE-lite
    h_prompt = f"""Escreva um parágrafo (4-6 linhas) que seria encontrado em acórdão jurídico
e responderia diretamente a esta pergunta. Use linguagem técnica formal.
Pergunta: {query_original}
Escreva APENAS o parágrafo."""
    hyde = llm_call(h_prompt, temperature=0.2, max_tokens=300) or query_original

    # Step-back
    s_prompt = f"""Dada a pergunta específica abaixo, formule uma pergunta MAIS GERAL
que abrange o conceito jurídico por trás dela.
Pergunta: {query_original}
Retorne APENAS a pergunta geral."""
    sback = llm_call(s_prompt, temperature=0.1, max_tokens=100) or query_original

    return {
        "original":     query_original,
        "paraphrase_1": paras[0],
        "paraphrase_2": paras[1],
        "hyde_lite":    hyde,
        "step_back":    sback,
    }


# === Pré-computa as 5 variantes para todas as 5 queries de teste ===
print(f"⏳ Gerando 5 variantes × {len(queries_baseline)} queries = {5*len(queries_baseline)} chamadas LLM…")
t0 = time.time()
resultados_rewriting = []
for q in queries_baseline:
    qid = q["id"]
    variantes = gerar_variantes(q["query_original"])
    resultados_rewriting.append({"id": qid, **variantes})
    print(f"   ✓ {qid}")
print(f"✅ Rewriting concluído em {time.time()-t0:.1f}s")
print(f"   {len(resultados_rewriting)} queries × 5 variantes")

---

## 🩺 Passo 10.5 — Diagnóstico Opcional: Gargalo + Sanity Check do Rerank

Esta célula faz **duas verificações** antes da matriz completa do Passo 11:

### A) Sanity Check do Reranker (NOVO em v6.2)

Testa se o reranker **discrimina relevante de irrelevante** com um par claro:
- Query: pergunta jurídica conhecida
- Doc relevante: artigo de lei correspondente
- Doc irrelevante: texto sobre tema completamente diferente

> ⚠️ Se a **diferença de scores for < 0.05**, o reranker está degenerado e a
> matriz completa vai dar resultados ruins. Alertaremos para você abortar.

### B) Diagnóstico de Performance

Cronometra 5 buscas em 3 estágios (embed query, hybrid search, rerank) para
detectar gargalos. Estima tempo total da matriz do Passo 11.

> 💡 Para **pular** esta célula, defina `SKIP_DIAGNOSTICO = True`.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 10.5 — Sanity Check (3 estratégias) + Diagnóstico Performance
# ════════════════════════════════════════════════════════════════════
SKIP_DIAGNOSTICO = False  # ← True = pula este passo
TEST_ALL_STRATEGIES = True  # ← True = sanity check rodando nas 3 estratégias

if SKIP_DIAGNOSTICO:
    print("ℹ️  Passo 10.5 pulado (SKIP_DIAGNOSTICO=True)")
else:
    # ═══════════════════════════════════════════════════════════════
    # PARTE A — Sanity Check do Reranker (NAS 3 ESTRATÉGIAS)
    # ═══════════════════════════════════════════════════════════════
    print("=" * 88)
    print("🧪 A) SANITY CHECK — Discriminação por estratégia")
    print("=" * 88)
    print()

    sanity_query   = "Quais os requisitos legais para decretação de prisão preventiva?"
    sanity_doc_rel = ("Art. 312 do CPP estabelece que a prisão preventiva poderá ser "
                       "decretada como garantia da ordem pública, da ordem econômica, "
                       "por conveniência da instrução criminal, ou para assegurar a "
                       "aplicação da lei penal, quando houver prova da existência do "
                       "crime e indício suficiente de autoria.")
    sanity_doc_irr = ("A guarda compartilhada de menores, regulada pela Lei 13.058/2014, "
                       "atende ao princípio do melhor interesse da criança. Ambos os "
                       "genitores exercem conjuntamente o poder familiar.")

    sanity_pair = [
        {"id":"doc_rel","texto":sanity_doc_rel,"fonte":"CPP Art. 312"},
        {"id":"doc_irr","texto":sanity_doc_irr,"fonte":"Lei 13.058"},
    ]

    estrategias_para_testar = (
        ["ollama_cosine","transformers_ce","flashrank_onnx"]
        if TEST_ALL_STRATEGIES else [RERANK_STRATEGY]
    )

    sanity_results = []
    for strat in estrategias_para_testar:
        print(f"  Testando estratégia: {strat}")
        try:
            t = time.time()
            ranked = rerank_caminho_a(sanity_query, sanity_pair, top_k=2, strategy=strat)
            elapsed = (time.time() - t) * 1000

            score_rel = next((d["score_reranker"] for d in ranked if d["id"]=="doc_rel"), None)
            score_irr = next((d["score_reranker"] for d in ranked if d["id"]=="doc_irr"), None)

            if score_rel is None or score_irr is None:
                veredito = "⚠️  N/D"
                delta = None
            else:
                delta = score_rel - score_irr
                if delta > 0.10:    veredito = "✅ EXCELENTE"
                elif delta > 0.05:  veredito = "✅ BOM"
                elif delta > 0.0:   veredito = "⚠️  FRACO"
                else:                veredito = "❌ INVERTIDO"

            sanity_results.append({
                "estratégia": strat,
                "score_rel": score_rel if score_rel is not None else 0,
                "score_irr": score_irr if score_irr is not None else 0,
                "Δ":         round(delta,4) if delta is not None else 0,
                "tempo_ms":  round(elapsed,0),
                "veredito":  veredito,
            })
        except Exception as e:
            print(f"     ❌ Falha: {e}")
            sanity_results.append({
                "estratégia": strat,
                "score_rel": 0, "score_irr": 0, "Δ": 0,
                "tempo_ms": 0, "veredito": "❌ ERRO",
            })

    df_sanity = pd.DataFrame(sanity_results)
    print()
    print(f"   Query:        \"{sanity_query[:60]}...\"")
    print(f"   Doc relevante: CPP Art. 312 (prisão preventiva)")
    print(f"   Doc irrelevante: Lei 13.058 (guarda compartilhada de menores)")
    print()
    print(df_sanity.to_string(index=False))
    print()
    print(f"💡 Interpretação:")
    print(f"   Δ > 0.10  ⇒ EXCELENTE discriminação (cross-encoder real funcionando bem)")
    print(f"   Δ > 0.05  ⇒ BOM (aceitável para produção)")
    print(f"   Δ > 0.0   ⇒ FRACO (pode funcionar, mas inconsistente)")
    print(f"   Δ ≤ 0.0   ⇒ INVERTIDO (rerank piora o ranking — não use)")

    # ═══════════════════════════════════════════════════════════════
    # PARTE B — Diagnóstico de Performance (estratégia ativa)
    # ═══════════════════════════════════════════════════════════════
    print()
    print("=" * 88)
    print(f"⏱  B) DIAGNÓSTICO DE PERFORMANCE — estratégia '{RERANK_STRATEGY}' (top-K={TOP_K_RETRIEVAL})")
    print("=" * 88)
    print()

    diag_rows = []
    for r in resultados_rewriting[:5]:
        qid   = r["id"]
        query = r["original"]

        t = time.time()
        q_vec = embeddings.embed_query(query)
        t_embed = (time.time() - t) * 1000

        t = time.time()
        body = _build_hybrid_query_clientside(query, q_vec, TOP_K_RETRIEVAL)
        try:
            resp = os_client.transport.perform_request(
                "POST", f"/{INDEX_NAME}/_search?search_pipeline={HYBRID_PIPELINE}",
                body=body
            )
            candidatos = [_hit_to_dict(h) for h in resp["hits"]["hits"]]
        except Exception as e:
            print(f"   ⚠️ {qid}: hybrid search falhou ({e})")
            candidatos = []
        t_search = (time.time() - t) * 1000

        t = time.time()
        reranked = rerank_caminho_a(query, candidatos, top_k=5) if candidatos else []
        t_rerank = (time.time() - t) * 1000

        t_total = t_embed + t_search + t_rerank
        pct_rerank = (t_rerank / t_total * 100) if t_total > 0 else 0

        diag_rows.append({
            "query_id":  qid,
            "embed_ms":  round(t_embed, 1),
            "search_ms": round(t_search, 1),
            "rerank_ms": round(t_rerank, 1),
            "total_ms":  round(t_total, 1),
            "%_rerank":  round(pct_rerank, 1),
        })

    df_diag = pd.DataFrame(diag_rows)
    print(df_diag.to_string(index=False))
    print()
    print(f"📊 Médias (estratégia '{RERANK_STRATEGY}'):")
    print(f"   Embed query  : {df_diag['embed_ms'].mean():>7.1f} ms")
    print(f"   Hybrid search: {df_diag['search_ms'].mean():>7.1f} ms")
    print(f"   Rerank       : {df_diag['rerank_ms'].mean():>7.1f} ms  ← {df_diag['%_rerank'].mean():.0f}% do tempo")
    print(f"   Total/busca  : {df_diag['total_ms'].mean():>7.1f} ms")
    print()
    tempo_estimado_s = df_diag["total_ms"].mean() * 100 / 1000 * 0.75
    print(f"📈 Projeção do Passo 11 (100 buscas): ~{tempo_estimado_s:.0f}s = {tempo_estimado_s/60:.1f} min")

---

## 📐 Passo 11 — Caminho A: Matriz 5 × 5 × 2 × 2 = 100 Buscas

Para isolar o efeito de cada técnica, rodamos **todas as combinações**:

- 5 queries do baseline
- 5 variantes de rewriting (`original` + 4 reescritas)
- 2 retrieval modes (`bm25`, `hybrid`)
- 2 rerank modes (`off`, `ollama`)

Total: **100 buscas** no caminho A com tracing LangFuse (tag `pipeline=client`).

> Cada linha do `df_completo` é depois classificada num dos 6 cenários (Passo 13).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Matriz exaustiva: 5 × 5 × 2 × 2 = 100 buscas no caminho A
# (C2 v6.1: latency visível no LangFuse via start_time/end_time)
# ════════════════════════════════════════════════════════════════════
import datetime

retrieval_modes_a = ["bm25", "hybrid"]
# === Valida disponibilidade da estratégia ativa ANTES da matriz ===
_strat_ok, _strat_msg = _strategy_disponivel(RERANK_STRATEGY)
rerank_modes_a    = ["off", "on"] if _strat_ok else ["off"]

variantes_nomes = ["original", "paraphrase_1", "paraphrase_2", "hyde_lite", "step_back"]
TOP_K_FINAL = 5

# === BANNER: confirma qual estratégia de rerank está ativa ===
print("=" * 78)
print(f"🎯 ESTRATÉGIA DE RERANK ATIVA: {RERANK_STRATEGY}")
print("=" * 78)
if RERANK_STRATEGY == "transformers_ce":
    print("   Cross-encoder REAL BAAI/bge-reranker-v2-m3 via HuggingFace transformers")
    print("   Qualidade: ⭐⭐⭐⭐⭐ | Tamanho: ~560 MB + torch")
elif RERANK_STRATEGY == "ollama_cosine":
    print("   Pseudo-rerank cosseno via Ollama bi-encoder")
    print("   Qualidade: ⭐⭐ | Pureza Ollama: ✅")
elif RERANK_STRATEGY == "flashrank_onnx":
    print("   Cross-encoder ONNX (ms-marco-MultiBERT-L-12)")
    print("   Qualidade: ⭐⭐⭐⭐ | Tamanho: ~150 MB | Sem torch")
print()
print(f"💡 Para mudar: ajuste RERANK_STRATEGY no Passo 8 ou no .env e rode de novo")
print(f"   Os 100 traces no LangFuse terão tag rerank=on:{RERANK_STRATEGY}")
print()

# === Confirma disponibilidade REAL da estratégia (não só do Ollama) ===
if _strat_ok:
    print(f"✅ Estratégia '{RERANK_STRATEGY}' DISPONÍVEL — rerank será executado")
    print(f"   {_strat_msg}")
else:
    print(f"⚠️  Estratégia '{RERANK_STRATEGY}' INDISPONÍVEL — rerank será PULADO")
    print(f"   Motivo: {_strat_msg}")
    print(f"   A matriz vai rodar 50 buscas (só sem rerank) em vez de 100.")
    print(f"   Para corrigir: ajuste RERANK_STRATEGY ou verifique a infra da estratégia escolhida.")
print()

linhas = []
total = len(resultados_rewriting) * len(variantes_nomes) * len(retrieval_modes_a) * len(rerank_modes_a)
print(f"⏳ Executando matriz A: {len(resultados_rewriting)} queries × {len(variantes_nomes)} variantes "
      f"× {len(retrieval_modes_a)} retrieval × {len(rerank_modes_a)} rerank = {total} buscas")

t0 = time.time()
for r in resultados_rewriting:
    qid  = r["id"]
    rels = ground_truth.get(qid, set())

    for v in variantes_nomes:
        texto = r[v]
        for rm in retrieval_modes_a:
            for rk in rerank_modes_a:
                t_inicio = datetime.datetime.now(datetime.timezone.utc)
                t_perf   = time.time()

                trace = lf_trace(
                    name=f"lab2-A-{rm}-{rk}",
                    tags=[f"retrieval={rm}", f"rerank={rk}:{RERANK_STRATEGY if rk == 'on' else 'none'}", v, qid, "pipeline=client"],
                    input={"query": texto, "variante": v, "retrieval_mode": rm,
                           "rerank_mode": rk, "pipeline": "client"},
                    start_time=t_inicio,
                )
                span_search = trace.span(name="search", start_time=t_inicio,
                                          input={"mode": rm, "top_k": TOP_K_RETRIEVAL})

                docs = buscar_corpus(texto, top_k=TOP_K_FINAL,
                                      retrieval_mode=rm, rerank_mode=rk)

                t_fim_perf = time.time()
                t_fim = datetime.datetime.now(datetime.timezone.utc)
                lat_ms = (t_fim_perf - t_perf) * 1000

                span_search.update(end_time=t_fim,
                                    output={"top_ids": [d["id"] for d in docs],
                                            "latency_ms": round(lat_ms, 1)})
                span_search.end(end_time=t_fim)

                ids = [d["id"] for d in docs]
                tp        = len(set(ids) & rels)
                recall    = tp / len(rels) if rels else 0.0
                precision = tp / TOP_K_FINAL
                mrr = 0.0
                for rank, doc_id in enumerate(ids, start=1):
                    if doc_id in rels: mrr = 1.0/rank; break

                trace.score(name="recall@5",    value=recall)
                trace.score(name="precision@5", value=precision)
                trace.score(name="mrr",         value=mrr)
                trace.update(
                    end_time=t_fim,
                    output={"top_ids": ids,
                            "recall@5":    round(recall, 3),
                            "precision@5": round(precision, 3),
                            "mrr":         round(mrr, 3),
                            "latency_ms":  round(lat_ms, 1)}
                )

                linhas.append({
                    "query_id":       qid,
                    "variante":       v,
                    "retrieval_mode": rm,
                    "rerank_mode":    rk,
                    "pipeline":       "A (client)",
                    "top5_ids":       ids,
                    "top5_fontes":    [d["fonte"] for d in docs],
                    "true_pos":       tp,
                    "recall@5":       round(recall, 3),
                    "precision@5":    round(precision, 3),
                    "mrr":            round(mrr, 3),
                    "latency_ms":     round(lat_ms, 1),
                })

if LANGFUSE_ON and langfuse: langfuse.flush()
elapsed = time.time() - t0
df_completo = pd.DataFrame(linhas)
print(f"✅ {len(df_completo)} buscas A em {elapsed:.1f}s ({elapsed/total*1000:.0f}ms/busca)")

---

## 🏁 Passo 11.5 — Mini-Comparativo: 3 Estratégias de Rerank no Cenário 5

Este passo executa o **cenário 5 (Advanced RAG completo)** com cada uma das 3
estratégias de rerank disponíveis, usando as 5 queries originais do baseline.

Total: **3 estratégias × 5 queries = 15 buscas extras** (~1-2 minutos).

> 💡 Para pular: defina `SKIP_RERANK_COMPARE = True` abaixo.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 11.5 — Mini-comparativo 3 estratégias × 5 queries (cenário 5)
# ════════════════════════════════════════════════════════════════════
SKIP_RERANK_COMPARE = False

if SKIP_RERANK_COMPARE:
    print("ℹ️  Passo 11.5 pulado (SKIP_RERANK_COMPARE=True)")
    df_rerank_compare = pd.DataFrame()
else:
    estrategias = ["ollama_cosine", "transformers_ce", "flashrank_onnx"]
    print(f"⏳ Comparativo: cenário 5 (Advanced RAG completo) × {len(estrategias)} estratégias × {len(resultados_rewriting)} queries")
    print(f"   Total: {len(estrategias) * len(resultados_rewriting)} buscas")
    print()

    rows = []
    for strat in estrategias:
        print(f"  ── Estratégia: {strat}")
        for r in resultados_rewriting:
            qid  = r["id"]
            rels = ground_truth.get(qid, set())
            # Cenário 5 usa: melhor rewriting + hybrid + rerank
            melhor_var = "paraphrase_1"  # padrão; será recalculado no Passo 13
            texto = r.get(melhor_var, r["original"])

            # Etapa 1: hybrid retrieval no caminho A
            q_vec = embeddings.embed_query(texto)
            body = _build_hybrid_query_clientside(texto, q_vec, TOP_K_RETRIEVAL)
            try:
                resp = os_client.transport.perform_request(
                    "POST", f"/{INDEX_NAME}/_search?search_pipeline={HYBRID_PIPELINE}",
                    body=body
                )
                candidatos = [_hit_to_dict(h) for h in resp["hits"]["hits"]]
            except Exception:
                candidatos = []

            # Etapa 2: rerank com a estratégia atual
            t = time.time()
            try:
                docs = rerank_caminho_a(texto, candidatos, top_k=TOP_K_FINAL, strategy=strat)
            except Exception as e:
                print(f"     ❌ {qid}/{strat}: {e}")
                docs = []
            lat = (time.time() - t) * 1000
            ids = [d["id"] for d in docs]

            tp     = len(set(ids) & rels)
            recall = tp / len(rels) if rels else 0.0
            mrr    = 0.0
            for rank, doc_id in enumerate(ids, start=1):
                if doc_id in rels: mrr = 1.0/rank; break

            rows.append({
                "estratégia": strat,
                "query_id":   qid,
                "recall@5":   round(recall, 3),
                "mrr":        round(mrr, 3),
                "latency_ms": round(lat, 1),
                "tp":         tp,
            })

    df_rerank_compare = pd.DataFrame(rows)

    # Agregação por estratégia
    resumo_rerank = (df_rerank_compare.groupby("estratégia", as_index=False)
                                       .agg(recall_medio=("recall@5","mean"),
                                            mrr_medio=("mrr","mean"),
                                            latency_media=("latency_ms","mean"))
                                       .round(3))
    # Marca o vencedor por métrica
    print()
    print("📊 COMPARAÇÃO POR ESTRATÉGIA (cenário 5 — Advanced RAG completo)")
    print("=" * 80)
    print(resumo_rerank.to_string(index=False))
    print()

    # Veredito
    best_recall = resumo_rerank.loc[resumo_rerank["recall_medio"].idxmax(), "estratégia"]
    best_latency = resumo_rerank.loc[resumo_rerank["latency_media"].idxmin(), "estratégia"]
    print(f"🏆 Melhor recall@5:    {best_recall}")
    print(f"⚡ Mais rápida:         {best_latency}")
    print()

    # Tabela colorida
    def cor_max(s, ascending=False):
        target = s.min() if ascending else s.max()
        return ["background-color: #c8e6c9; font-weight: bold" if v == target else "" for v in s]

    display(
        resumo_rerank.style
            .apply(lambda s: cor_max(s, ascending=False), subset=["recall_medio","mrr_medio"])
            .apply(lambda s: cor_max(s, ascending=True), subset=["latency_media"])
            .format({"recall_medio":"{:.3f}","mrr_medio":"{:.3f}","latency_media":"{:.0f}"})
            .set_caption("3 estratégias × cenário 5 — verde = melhor")
    ) if 'display' in dir() else None

    print("💡 Use estes resultados para decidir o RERANK_STRATEGY no Passo 8")
    print("   antes de rodar a matriz completa do Passo 11 (100 buscas).")

---

## 🎨 Passo 12 — Caminho A: Tabela Colorida das 100 Buscas

Cada linha mostra uma combinação `(query, variante, retrieval, rerank)`.

🟢 ≥3 acertos | 🟡 1-2 acertos | 🔴 0 acertos


In [ ]:
def colorir_acertos(row):
    n = row["true_pos"]
    cor = "#c8e6c9" if n >= 3 else ("#fff9c4" if n >= 1 else "#ffcdd2")
    return [f"background-color: {cor}"] * len(row)

df_visu = df_completo[[
    "query_id","variante","retrieval_mode","rerank_mode",
    "true_pos","recall@5","precision@5","mrr","latency_ms","top5_fontes"
]].copy()
df_visu["top5_fontes"] = df_visu["top5_fontes"].apply(lambda x: " | ".join(x))
df_visu = df_visu.sort_values(["query_id","variante","retrieval_mode","rerank_mode"]).reset_index(drop=True)

(df_visu.style
   .apply(colorir_acertos, axis=1)
   .format({"recall@5":"{:.2f}","precision@5":"{:.2f}","mrr":"{:.2f}","latency_ms":"{:.0f}"})
   .set_properties(**{"font-size":"10px","color":"black"})
   .set_table_styles([{"selector":"th","props":[("background-color","#37474f"),("color","white")]}]))

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Classifica cada linha num dos 6 cenários de ablação
# ════════════════════════════════════════════════════════════════════
# Convenção:
#   variante == "original"  ⇒ SEM rewriting
#   variante != "original"  ⇒ COM rewriting (usa a MELHOR variante)
#   retrieval_mode == "bm25"   ⇒ SEM hybrid (linha base)
#   retrieval_mode == "hybrid" ⇒ COM hybrid
#   rerank_mode == "off"    ⇒ SEM rerank
#   rerank_mode != "off"    ⇒ COM rerank

CENARIOS_NOMES = {
    0: "0. Baseline (BM25 puro)",
    1: "1. + Rewriting",
    2: "2. + Hybrid",
    3: "3. + Rerank",
    4: "4. + Hybrid + Rerank",
    5: "5. Advanced RAG completo",
}

def classificar_cenario_row(row, melhor_variante: str) -> int | None:
    """Mapeia (variante, retrieval, rerank) → cenário 0-5.

    Apenas linhas que CASAM com um cenário canônico são retornadas;
    demais combinações ficam None.
    """
    is_rewriting = row["variante"] == melhor_variante and row["variante"] != "original"
    is_original  = row["variante"] == "original"
    is_hybrid    = row["retrieval_mode"] == "hybrid"
    is_bm25      = row["retrieval_mode"] == "bm25"
    is_rerank    = row["rerank_mode"] != "off"
    is_no_rerank = row["rerank_mode"] == "off"

    if is_original    and is_bm25    and is_no_rerank: return 0  # baseline
    if is_rewriting   and is_bm25    and is_no_rerank: return 1  # + rewriting
    if is_original    and is_hybrid  and is_no_rerank: return 2  # + hybrid
    if is_original    and is_bm25    and is_rerank:    return 3  # + rerank
    if is_original    and is_hybrid  and is_rerank:    return 4  # + hybrid + rerank
    if is_rewriting   and is_hybrid  and is_rerank:    return 5  # Advanced RAG completo
    return None  # combinação intermediária (existe na matriz mas não rotulamos)


def aplicar_cenarios(df: pd.DataFrame, melhor_variante: str) -> pd.DataFrame:
    """Adiciona coluna 'cenario' classificando cada linha (ou None)."""
    df = df.copy()
    df["cenario"] = df.apply(lambda r: classificar_cenario_row(r, melhor_variante), axis=1)
    df["cenario_nome"] = df["cenario"].map(CENARIOS_NOMES)
    return df


def resumo_cenarios(df: pd.DataFrame) -> pd.DataFrame:
    """Agrupa por cenário e calcula média das 4 métricas."""
    sub = df[df["cenario"].notna()].copy()
    agg = (sub.groupby(["cenario", "cenario_nome"], as_index=False)
              .agg(recall_5_medio=("recall@5",     "mean"),
                   precision_5_medio=("precision@5", "mean"),
                   mrr_medio=("mrr",               "mean"),
                   latency_medio=("latency_ms",   "mean"))
              .sort_values("cenario"))
    for c in ["recall_5_medio","precision_5_medio","mrr_medio"]:
        agg[c] = agg[c].round(3)
    agg["latency_medio"] = agg["latency_medio"].round(0)
    baseline = agg.iloc[0]
    agg["Δ_recall"]    = (agg["recall_5_medio"]    - baseline["recall_5_medio"]).round(3)
    agg["Δ_precision"] = (agg["precision_5_medio"] - baseline["precision_5_medio"]).round(3)
    agg["Δ_mrr"]       = (agg["mrr_medio"]         - baseline["mrr_medio"]).round(3)
    agg["Δ_latency"]   = (agg["latency_medio"]     - baseline["latency_medio"]).round(0)
    return agg

print("✅ Helpers classificar_cenario_row() e resumo_cenarios() prontos")

---

## 🏆 Passo 13 — Caminho A: 6 Cenários de Ablação

Cada cenário isola UMA técnica vs baseline. As colunas Δ mostram o **ganho atribuível** a
cada técnica isoladamente.

> 💡 Para o cenário "+Rewriting" usamos a **melhor variante** identificada pelo recall
> médio sobre as queries (escolhida automaticamente abaixo).


In [ ]:
# === Identificar a melhor variante de rewriting (média recall com BM25 puro) ===
variantes_recall = (
    df_completo[(df_completo["variante"] != "original") &
                 (df_completo["retrieval_mode"] == "bm25") &
                 (df_completo["rerank_mode"] == "off")]
    .groupby("variante")["recall@5"].mean().round(3).sort_values(ascending=False)
)
print("📊 Recall@5 médio por variante de rewriting (BM25 puro):")
print(variantes_recall.to_string())
melhor_variante_a = variantes_recall.idxmax()
print(f"\\n📌 Melhor variante (caminho A): '{melhor_variante_a}'")

# === Classificar cenários e agregar ===
df_completo = aplicar_cenarios(df_completo, melhor_variante_a)
resumo_a = resumo_cenarios(df_completo)
print()
print("📋 6 CENÁRIOS DE ABLAÇÃO — CAMINHO A:")
print("=" * 100)
print(resumo_a[["cenario_nome","recall_5_medio","precision_5_medio","mrr_medio","latency_medio",
                 "Δ_recall","Δ_precision","Δ_mrr","Δ_latency"]].to_string(index=False))
print()

def cor_pos(v):
    return "background-color: #c8e6c9; font-weight: bold" if v>0 else \
           ("background-color: #ffcdd2" if v<0 else "")
def cor_neg(v):
    return "background-color: #c8e6c9; font-weight: bold" if v<0 else \
           ("background-color: #ffcdd2" if v>0 else "")

(resumo_a[["cenario_nome","recall_5_medio","precision_5_medio","mrr_medio","latency_medio",
            "Δ_recall","Δ_precision","Δ_mrr","Δ_latency"]]
    .style
    .map(cor_pos, subset=["Δ_recall","Δ_precision","Δ_mrr"])
    .map(cor_neg, subset=["Δ_latency"])
    .format({"recall_5_medio":"{:.2f}","precision_5_medio":"{:.2f}","mrr_medio":"{:.2f}",
             "Δ_recall":"{:+.2f}","Δ_precision":"{:+.2f}","Δ_mrr":"{:+.2f}",
             "latency_medio":"{:.0f}","Δ_latency":"{:+.0f}"})
    .set_caption(f"Caminho A — Ablação dos 6 cenários (melhor rewriting: {melhor_variante_a})"))

---

## 📈 Passo 14 — Caminho A: Barplot de Ablação


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metricas = [("recall_5_medio","Recall@5"),
             ("precision_5_medio","Precision@5"),
             ("mrr_medio","MRR")]
labels6 = [n.split(". ",1)[1][:18] for n in resumo_a["cenario_nome"]]
cores6  = ["#90a4ae", "#5e35b1", "#1e88e5", "#43a047", "#fb8c00", "#c62828"]
x = np.arange(len(labels6))

for ax, (col, titulo) in zip(axes, metricas):
    vals = resumo_a[col].values
    ax.bar(x, vals, color=cores6, edgecolor="black", width=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels6, rotation=25, ha="right", fontsize=8)
    ax.set_title(f"{titulo} — Caminho A", fontweight="bold")
    ax.set_ylim(0, max(vals.max()*1.25, 0.05))
    ax.axhline(vals[0], color="#90a4ae", linestyle="--", alpha=0.6,
               label=f"baseline ({vals[0]:.2f})")
    for i,v in enumerate(vals):
        ax.text(i, v+0.005, f"{v:.2f}", ha="center", fontweight="bold", fontsize=8)
    ax.legend(loc="upper left", fontsize=7); ax.grid(axis="y", alpha=0.3)

plt.suptitle(f"Caminho A — Ablação por cenário (Ollama + OpenSearch storage)",
             fontsize=11, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig("/tmp/lab2_v6_ablacao_A.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Gráfico A salvo: /tmp/lab2_v6_ablacao_A.png")

---

# 🅱️ PARTE B — Caminho Server-side OpenSearch

Agora a inversão completa: o OpenSearch absorve TUDO — embedding, hybrid e rerank.
Python só dispara queries `neural`.

| Recurso | Caminho A | **Caminho B** |
|---|---|---|
| Modelo de embedding | Ollama bge-m3 (Python) | **ml-commons all-MiniLM-L12-v2** |
| Geração de vetor (índice) | Python computa | **OpenSearch via ingest pipeline** |
| Geração de vetor (busca) | Python computa | **OpenSearch via `neural` query** |
| Hybrid search | hybrid pipeline + vetor pré-calc | **hybrid pipeline + `neural` query** |
| Reranker | Ollama qllama/bge-reranker-v2-m3 | **ml-commons ms-marco-MiniLM-L-12-v2** |

> ⚠️ Esta parte só executa se o plugin `opensearch-ml` estiver disponível (verificado no Passo 2).


---

## 🌐 Passo 16 — Caminho B: Registrar Modelo de Embedding no ml-commons

A partir daqui começamos o **caminho B**: deixar o OpenSearch fazer TUDO (embedding,
indexação automática, busca semântica). Vai funcionar em paralelo ao caminho A que
você já indexou no Passo 5A — sem alterar nada do que já existe.

### O que vamos fazer

1. Criar um **model group** dedicado ao LAB2
2. Registrar `huggingface/sentence-transformers/all-MiniLM-L12-v2` (384 dims, ~120 MB)
3. Esperar o registro completar (polling do `task_id`)
4. Guardar o `EMBED_MODEL_ID_OS` para usar no ingest pipeline e na neural query

### Por que `all-MiniLM-L12-v2` e não o `bge-m3`?

| Modelo | Dims | Tamanho | Catálogo ml-commons? | Multi-língua | Trade-off |
|---|---|---|---|---|---|
| `all-MiniLM-L12-v2` | 384 | ~120 MB | ✅ pretrained | Limitado | Leve, **pronto pra usar** sem upload |
| `bge-m3` (Aula 1) | 1024 | ~2 GB | ❌ exige upload manual | Forte | Melhor PT-BR mas burocrático |

O objetivo aqui é **arquitetural**: mostrar QUE o OpenSearch pode gerar embeddings
server-side. Em produção, basta substituir o modelo (upload de `.pt` + register).

📖 Doc: <https://docs.opensearch.org/latest/ml-commons-plugin/pretrained-models/>


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 5C — Registrar all-MiniLM-L12-v2 no ml-commons
# ════════════════════════════════════════════════════════════════════
TIMEOUT_REGISTRO = int(os.getenv("TIMEOUT_REGISTRO", "120"))  # segundos para esperar deploy
NEURAL_EMBED_DIM = 384
NEURAL_INDEX     = f"{INDEX_NAME}_neural"
NEURAL_MODEL_NAME = "huggingface/sentence-transformers/all-MiniLM-L12-v2"
NEURAL_MODEL_VER  = "1.0.2"

HAS_NEURAL_INDEX = False  # vira True se o setup completo do caminho B funcionar
EMBED_MODEL_ID_OS = None

if not (USE_OPENSEARCH and HAS_ML_COMMONS):
    print("ℹ️  Caminho B desabilitado (precisa OpenSearch + plugin ml-commons)")
    print("   Você ainda pode rodar o lab inteiro pelo caminho A.")
else:
    try:
        # === 1) Ajustar cluster settings p/ permitir registro do catálogo ===
        os_client.cluster.put_settings(body={
            "persistent": {
                "plugins.ml_commons.only_run_on_ml_node":            "false",
                "plugins.ml_commons.allow_registering_model_via_url":"true",
                "plugins.ml_commons.native_memory_threshold":         99,
                "plugins.ml_commons.model_access_control_enabled":    "true",
            }
        })
        print("✅ Cluster settings ajustados (ml_commons)")

        # === 2) Criar (ou recuperar) model group dedicado ===
        try:
            grp = os_client.transport.perform_request(
                "POST", "/_plugins/_ml/model_groups/_register",
                body={"name": "lab2-neural-embed",
                      "description": "Modelos do caminho B (LAB2 — Neural Search)"}
            )
            embed_group = grp["model_group_id"]
            print(f"✅ Model group: {embed_group}")
        except Exception as e:
            print(f"ℹ️  Group já existia ou falhou registro: {e}")
            embed_group = None

        # === 3) Registrar all-MiniLM-L12-v2 com deploy automático ===
        reg_body = {
            "name":           NEURAL_MODEL_NAME,
            "version":        NEURAL_MODEL_VER,
            "model_format":   "TORCH_SCRIPT",
        }
        if embed_group:
            reg_body["model_group_id"] = embed_group

        reg = os_client.transport.perform_request(
            "POST", "/_plugins/_ml/models/_register?deploy=true",
            body=reg_body,
        )
        task_id = reg.get("task_id")
        print(f"⏳ Registrando {NEURAL_MODEL_NAME} (task {task_id})…")
        print(f"   Timeout: {TIMEOUT_REGISTRO}s | polling a cada 2s")

        # === 4) Polling do task até COMPLETED ===
        deadline = time.time() + TIMEOUT_REGISTRO
        while time.time() < deadline:
            status = os_client.transport.perform_request("GET", f"/_plugins/_ml/tasks/{task_id}")
            state = status.get("state", "UNKNOWN")
            if state == "COMPLETED":
                EMBED_MODEL_ID_OS = status.get("model_id")
                print(f"✅ Modelo deployado: {EMBED_MODEL_ID_OS}")
                HAS_NEURAL_INDEX = True
                break
            elif state == "FAILED":
                print(f"❌ Registro falhou: {status.get('error','desconhecido')}")
                break
            time.sleep(2)
        else:
            print(f"⏱  Timeout após {TIMEOUT_REGISTRO}s. Task ainda em {state}.")
            print(f"   Aumente TIMEOUT_REGISTRO ou verifique logs do cluster.")

    except Exception as e:
        print(f"⚠️  Falha ao registrar modelo: {e}")
        print(f"   Caminho B desabilitado, caminho A continua disponível.")
        HAS_NEURAL_INDEX = False

---

## 📦 Passo 17 — Caminho B: Ingest Pipeline + Índice Paralelo + Índice Paralelo

Com o modelo registrado, criamos a infraestrutura para que **o OpenSearch gere os
embeddings automaticamente** durante a indexação:

1. **Ingest pipeline `lab2-neural-ingest`** com `text_embedding` processor
2. **Índice `corpus_juridico_neural`** com `default_pipeline` apontando para o ingest
3. **Bulk dos 80 documentos sem campo `embedding`** — o processor preenche

### Diferença chave vs. Passo 5A

```
Caminho A (Passo 5A)                    Caminho B (este passo)
─────────────────────                   ──────────────────────
1. Python gera embedding                 1. PUT ingest pipeline
   embeddings.embed_documents(txt)         (text_embedding processor)
2. bulk inclui "embedding": [...]        2. PUT índice com default_pipeline
3. OpenSearch só armazena                3. bulk SEM "embedding": [...]
                                         4. OpenSearch chama o modelo
                                            e gera o embedding no servidor
```


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 5D — Pipeline de ingestão + índice neural
# ════════════════════════════════════════════════════════════════════
NEURAL_INGEST_PIPELINE = "lab2-neural-ingest"

if not HAS_NEURAL_INDEX:
    print("ℹ️  Passo 5D pulado (modelo embed não está deployado)")
else:
    try:
        # === 1) Criar ingest pipeline com text_embedding processor ===
        # Doc: field_map mapeia campo de origem → campo de destino do vetor
        ingest_body = {
            "description": "Gera embedding automático da ementa+texto ao indexar",
            "processors": [
                {
                    "text_embedding": {
                        "model_id": EMBED_MODEL_ID_OS,
                        "field_map": {
                            "texto": "embedding"  # input: 'texto' → output: 'embedding'
                        }
                    }
                }
            ]
        }
        os_client.transport.perform_request(
            "PUT", f"/_ingest/pipeline/{NEURAL_INGEST_PIPELINE}",
            body=ingest_body
        )
        print(f"✅ Ingest pipeline '{NEURAL_INGEST_PIPELINE}' criado")
        print(f"   Origem: campo 'texto' → Destino: campo 'embedding' (knn_vector)")

        # === 2) Criar índice paralelo com default_pipeline ===
        neural_mapping = {
            "settings": {
                "index": {
                    "knn": True,
                    "default_pipeline": NEURAL_INGEST_PIPELINE,
                }
            },
            "mappings": {
                "properties": {
                    "texto":          {"type": "text", "analyzer": "portuguese"},
                    "ementa":         {"type": "text", "analyzer": "portuguese"},
                    "palavras_chave": {"type": "keyword"},
                    "tipo":           {"type": "keyword"},
                    "tribunal":       {"type": "keyword"},
                    "numero":         {"type": "keyword"},
                    "data":           {"type": "date", "ignore_malformed": True},
                    "embedding": {
                        "type": "knn_vector",
                        "dimension": NEURAL_EMBED_DIM,
                        "method": {
                            "name": "hnsw",
                            "engine": "faiss",
                            "space_type": "cosinesimil",
                            "parameters": {"ef_construction": 128, "m": 16}
                        }
                    },
                }
            }
        }

        if os_client.indices.exists(index=NEURAL_INDEX):
            print(f"ℹ️  Índice '{NEURAL_INDEX}' já existia — recriando")
            os_client.indices.delete(index=NEURAL_INDEX)

        os_client.indices.create(index=NEURAL_INDEX, body=neural_mapping)
        print(f"✅ Índice '{NEURAL_INDEX}' criado (dim={NEURAL_EMBED_DIM}, pipeline auto)")

        # === 3) Bulk SEM embedding — o processor gera no servidor ===
        from opensearchpy.helpers import bulk as os_bulk
        print(f"⏳ Indexando {len(documentos)} docs (OpenSearch gera os embeddings)…")
        t0 = time.time()
        actions = [
            {
                "_index": NEURAL_INDEX,
                "_id":    d["id"],
                "_source": {
                    # ATENÇÃO: NÃO incluir 'embedding' — o processor preenche
                    "texto":          d.get("texto", ""),
                    "ementa":         d.get("ementa", ""),
                    "palavras_chave": d.get("palavras_chave", []),
                    "tipo":           d.get("tipo", ""),
                    "tribunal":       d.get("tribunal", ""),
                    "numero":         d.get("numero", d["id"]),
                    "data":           d.get("data", ""),
                },
            }
            for d in documentos
        ]
        success, errors = os_bulk(os_client, actions, refresh="wait_for")
        elapsed = time.time() - t0
        n_errors = len(errors) if isinstance(errors, list) else errors
        print(f"✅ Bulk concluído em {elapsed:.1f}s: {success} indexados, {n_errors} erros")

        count = os_client.count(index=NEURAL_INDEX)["count"]
        print(f"   Total no índice '{NEURAL_INDEX}': {count}")

        # === 4) Verificar que o embedding FOI gerado pelo servidor ===
        sample = os_client.get(index=NEURAL_INDEX, id="doc_001",
                                _source=["embedding"])
        embed_len = len(sample["_source"].get("embedding", []))
        if embed_len == NEURAL_EMBED_DIM:
            print(f"✅ Confirmado: embedding dim={embed_len} gerado pelo OpenSearch server-side")
        else:
            print(f"⚠️  Embedding inesperado: dim={embed_len} (esperado {NEURAL_EMBED_DIM})")

    except Exception as e:
        print(f"❌ Falha no Passo 5D: {e}")
        HAS_NEURAL_INDEX = False

---

## 🎯 Passo 18 — Caminho B: Registrar Cross-Encoder no ml-commons

Diferente do caminho A (que usa Ollama), o caminho B usa o **cross-encoder oficial
do catálogo ml-commons**: `huggingface/cross-encoders/ms-marco-MiniLM-L-12-v2`.

📖 Doc: <https://docs.opensearch.org/latest/ml-commons-plugin/pretrained-models/#cross-encoder-models>


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Registrar cross-encoder no ml-commons (caminho B)
# ════════════════════════════════════════════════════════════════════
RERANK_MODEL_ID  = None
RERANK_OS_OK     = False

if not HAS_NEURAL_INDEX:
    print("ℹ️  Passo 18 pulado (caminho B inativo)")
else:
    try:
        # Recupera ou cria model group dedicado ao rerank
        try:
            grp = os_client.transport.perform_request(
                "POST", "/_plugins/_ml/model_groups/_register",
                body={"name": "lab2-rerank", "description": "Cross-encoder LAB2 caminho B"}
            )
            rerank_group = grp["model_group_id"]
        except Exception:
            rerank_group = None

        reg_body = {
            "name":         "huggingface/cross-encoders/ms-marco-MiniLM-L-12-v2",
            "version":      "1.0.2",
            "model_format": "TORCH_SCRIPT",
        }
        if rerank_group:
            reg_body["model_group_id"] = rerank_group

        reg = os_client.transport.perform_request(
            "POST", "/_plugins/_ml/models/_register?deploy=true",
            body=reg_body
        )
        task_id = reg.get("task_id")
        print(f"⏳ Registrando cross-encoder (task {task_id})…")

        deadline = time.time() + TIMEOUT_REGISTRO
        while time.time() < deadline:
            status = os_client.transport.perform_request("GET", f"/_plugins/_ml/tasks/{task_id}")
            state = status.get("state", "UNKNOWN")
            if state == "COMPLETED":
                RERANK_MODEL_ID = status.get("model_id")
                print(f"✅ Cross-encoder deployado: {RERANK_MODEL_ID}")
                RERANK_OS_OK = True
                break
            elif state == "FAILED":
                print(f"❌ Registro falhou: {status.get('error','unknown')}")
                break
            time.sleep(2)
    except Exception as e:
        print(f"⚠️  Falha ao registrar cross-encoder: {e}")
        RERANK_OS_OK = False

---

## 🔗 Passo 19 — Caminho B: Search Pipelines (Hybrid + Rerank) (Hybrid + Rerank Neural)

Pares dos pipelines do caminho A, mas usando o **`model_id` do ml-commons** em vez
de campos pré-calculados. Dois novos pipelines:

| Pipeline                       | Uso                                                |
|--------------------------------|----------------------------------------------------|
| `lab2-neural-hybrid-pipeline`  | normalization-processor BM25+kNN (igual ao A)      |
| `lab2-neural-rerank-pipeline`  | rerank-processor com cross-encoder ml-commons      |

📖 Doc rerank: <https://docs.opensearch.org/latest/search-plugins/search-relevance/rerank-cross-encoder/>


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Passo 7B — Pipelines de busca do caminho B
# ════════════════════════════════════════════════════════════════════
NEURAL_HYBRID_PIPELINE = "lab2-neural-hybrid-pipeline"
NEURAL_RERANK_PIPELINE = "lab2-neural-rerank-pipeline"
NEURAL_RERANK_OK       = False

if not HAS_NEURAL_INDEX:
    print("ℹ️  Passo 7B pulado (índice neural não disponível)")
else:
    try:
        # === 1) Hybrid pipeline (idêntico ao caminho A) ===
        os_client.transport.perform_request(
            "PUT", f"/_search/pipeline/{NEURAL_HYBRID_PIPELINE}",
            body={
                "description": "Hybrid BM25 + kNN (caminho B / neural)",
                "phase_results_processors": [
                    {
                        "normalization-processor": {
                            "normalization": {"technique": "min_max"},
                            "combination": {
                                "technique": "arithmetic_mean",
                                "parameters": {"weights": [0.3, 0.7]}
                            }
                        }
                    }
                ]
            }
        )
        print(f"✅ Search pipeline '{NEURAL_HYBRID_PIPELINE}' criado")

        # === 2) Rerank pipeline — só se o modelo rerank do Passo 7 foi registrado ===
        if RERANK_OS_OK and RERANK_MODEL_ID:
            os_client.transport.perform_request(
                "PUT", f"/_search/pipeline/{NEURAL_RERANK_PIPELINE}",
                body={
                    "description": "Rerank com cross-encoder no caminho neural",
                    "response_processors": [
                        {
                            "rerank": {
                                "ml_opensearch": {"model_id": RERANK_MODEL_ID},
                                "context": {"document_fields": ["texto", "ementa"]}
                            }
                        }
                    ]
                }
            )
            print(f"✅ Search pipeline '{NEURAL_RERANK_PIPELINE}' criado")
            NEURAL_RERANK_OK = True
        else:
            print(f"⚠️  Rerank model do Passo 7 não disponível — pulando rerank pipeline neural")

    except Exception as e:
        print(f"⚠️  Falha em 7B: {e}")
        NEURAL_RERANK_OK = False

---

## 🔮 Passo 20 — Caminho B: `buscar_corpus_neural()` com 4 Combinações

Mesma assinatura do `buscar_corpus()` do caminho A, mas usando query `neural`.

| Parâmetro | Valores |
|---|---|
| `retrieval_mode` | `"bm25"` (sem vetor) ou `"hybrid"` (BM25 + neural) |
| `rerank_mode`    | `"off"` ou `"ml_commons"` |


In [ ]:
# ════════════════════════════════════════════════════════════════════
# buscar_corpus_neural() — caminho B com 4 combinações
# ════════════════════════════════════════════════════════════════════

def _build_bm25_neural(query: str, size: int) -> dict:
    """BM25 puro no índice neural (linha base do caminho B)."""
    return {
        "size": size,
        "query": {
            "multi_match": {
                "query":  query,
                "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                "type":   "best_fields",
            }
        }
    }


def _build_hybrid_neural(query: str, size: int) -> dict:
    """Hybrid no caminho B: match BM25 + neural (OpenSearch gera o vetor)."""
    return {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "multi_match": {
                            "query":  query,
                            "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                            "type":   "best_fields",
                        }
                    },
                    {
                        "neural": {
                            "embedding": {
                                "query_text": query,
                                "model_id":   EMBED_MODEL_ID_OS,
                                "k":          size,
                            }
                        }
                    }
                ]
            }
        }
    }


def buscar_corpus_neural(query: str, top_k: int = 5,
                          retrieval_mode: str = "bm25",
                          rerank_mode: str = "off") -> list:
    """Busca no índice corpus_juridico_neural (caminho B — server-side puro).

    Args:
        retrieval_mode : "bm25" | "hybrid"
        rerank_mode    : "off" | "ml_commons"
    """
    if not HAS_NEURAL_INDEX:
        return []

    candidate_size = TOP_K_RETRIEVAL if rerank_mode != "off" else top_k

    if retrieval_mode == "bm25":
        body = _build_bm25_neural(query, candidate_size)
    elif retrieval_mode == "hybrid":
        body = _build_hybrid_neural(query, candidate_size)
    else:
        raise ValueError(f"retrieval_mode desconhecido: {retrieval_mode}")

    # === Etapa 1: retrieval ===
    if rerank_mode == "off":
        try:
            if retrieval_mode == "hybrid":
                resp = os_client.transport.perform_request(
                    "POST", f"/{NEURAL_INDEX}/_search?search_pipeline={NEURAL_HYBRID_PIPELINE}",
                    body=body
                )
            else:
                resp = os_client.search(index=NEURAL_INDEX, body=body)
        except Exception as e:
            print(f"⚠️  Busca neural ({retrieval_mode}) falhou: {e}")
            return []
        return [_hit_to_dict(h) for h in resp["hits"]["hits"]]

    # === Etapa 2: rerank server-side ===
    if rerank_mode == "ml_commons":
        if not (RERANK_OS_OK and NEURAL_RERANK_OK):
            print(f"⚠️  rerank ml_commons pedido mas não disponível — devolvendo retrieval puro")
            return buscar_corpus_neural(query, top_k=top_k,
                                         retrieval_mode=retrieval_mode, rerank_mode="off")
        # Aqui que o RERANK acontece no Opensearch                                 
        body["ext"] = {"rerank": {"query_context": {"query_text": query}}}
        try:
            if retrieval_mode == "hybrid":
                resp = os_client.transport.perform_request(
                    "POST",
                    f"/{NEURAL_INDEX}/_search?search_pipeline={NEURAL_RERANK_PIPELINE}",
                    body=body
                )
            else:
                resp = os_client.transport.perform_request(
                    "POST", f"/{NEURAL_INDEX}/_search?search_pipeline={NEURAL_RERANK_PIPELINE}",
                    body=body
                )
            return [_hit_to_dict(h) for h in resp["hits"]["hits"][:top_k]]
        except Exception as e:
            print(f"⚠️  Rerank neural falhou: {e}")
            return []

    raise ValueError(f"rerank_mode desconhecido: {rerank_mode}")


print(f"✅ Função buscar_corpus_neural() pronta (caminho B — server-side puro)")
print(f"   retrieval_mode: bm25 | hybrid")
print(f"   rerank_mode:    off | ml_commons  ({'ON' if RERANK_OS_OK else 'OFF'})")
print(f"   Combinações:    4")

---

## 📐 Passo 21 — Caminho B: Matriz Simétrica 5 × 5 × 2 × 2 = 100 Buscas

Mesma estrutura do Passo 11 (caminho A), agora server-side. Reaproveita
`resultados_rewriting` do Passo 10 — variantes idênticas.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Matriz simétrica caminho B (C2 v6.1: latency visível no LangFuse)
# ════════════════════════════════════════════════════════════════════
import datetime as _dt

linhas_neural = []

if not HAS_NEURAL_INDEX:
    print("ℹ️  Caminho B inativo — matriz B pulada")
    df_neural = pd.DataFrame()
else:
    retrieval_modes_b = ["bm25", "hybrid"]
    rerank_modes_b    = ["off", "ml_commons"] if RERANK_OS_OK else ["off"]

    total = len(resultados_rewriting) * len(variantes_nomes) * len(retrieval_modes_b) * len(rerank_modes_b)
    print(f"⏳ Executando matriz B: {total} buscas…")
    t0 = time.time()
    for r in resultados_rewriting:
        qid  = r["id"]
        rels = ground_truth.get(qid, set())
        for v in variantes_nomes:
            texto = r[v]
            for rm in retrieval_modes_b:
                for rk in rerank_modes_b:
                    t_inicio = _dt.datetime.now(_dt.timezone.utc)
                    t_perf   = time.time()

                    trace = lf_trace(
                        name=f"lab2-B-{rm}-{rk}",
                        tags=[f"retrieval={rm}", f"rerank={rk}", v, qid, "pipeline=neural"],
                        input={"query": texto, "variante": v, "retrieval_mode": rm,
                               "rerank_mode": rk, "pipeline": "neural"},
                        start_time=t_inicio,
                    )
                    span = trace.span(name="search", start_time=t_inicio,
                                       input={"mode": rm, "top_k": TOP_K_RETRIEVAL})

                    docs = buscar_corpus_neural(texto, top_k=TOP_K_FINAL,
                                                  retrieval_mode=rm, rerank_mode=rk)

                    t_fim_perf = time.time()
                    t_fim = _dt.datetime.now(_dt.timezone.utc)
                    lat = (t_fim_perf - t_perf) * 1000

                    span.update(end_time=t_fim,
                                 output={"top_ids": [d["id"] for d in docs],
                                         "latency_ms": round(lat, 1)})
                    span.end(end_time=t_fim)

                    ids = [d["id"] for d in docs]
                    tp        = len(set(ids) & rels)
                    recall    = tp / len(rels) if rels else 0.0
                    precision = tp / TOP_K_FINAL
                    mrr = 0.0
                    for rank, doc_id in enumerate(ids, start=1):
                        if doc_id in rels: mrr = 1.0/rank; break

                    trace.score(name="recall@5",    value=recall)
                    trace.score(name="precision@5", value=precision)
                    trace.score(name="mrr",         value=mrr)
                    trace.update(
                        end_time=t_fim,
                        output={"top_ids": ids,
                                "recall@5":    round(recall, 3),
                                "precision@5": round(precision, 3),
                                "mrr":         round(mrr, 3),
                                "latency_ms":  round(lat, 1)}
                    )

                    linhas_neural.append({
                        "query_id":       qid,
                        "variante":       v,
                        "retrieval_mode": rm,
                        "rerank_mode":    rk,
                        "pipeline":       "B (neural)",
                        "top5_ids":       ids,
                        "top5_fontes":    [d["fonte"] for d in docs],
                        "true_pos":       tp,
                        "recall@5":       round(recall, 3),
                        "precision@5":    round(precision, 3),
                        "mrr":            round(mrr, 3),
                        "latency_ms":     round(lat, 1),
                    })
    if LANGFUSE_ON and langfuse: langfuse.flush()
    elapsed = time.time() - t0
    df_neural = pd.DataFrame(linhas_neural)
    print(f"✅ {len(df_neural)} buscas B em {elapsed:.1f}s")

---

## 🎨 Passo 22 — Caminho B: Tabela Colorida + Cenários

🟢 ≥3 acertos | 🟡 1-2 acertos | 🔴 0 acertos


In [ ]:
if df_neural.empty:
    print("ℹ️  Sem dados B")
    melhor_variante_b = "original"
    resumo_b = pd.DataFrame()
else:
    # Tabela colorida
    df_visu_b = df_neural[["query_id","variante","retrieval_mode","rerank_mode",
                             "true_pos","recall@5","precision@5","mrr","latency_ms","top5_fontes"]].copy()
    df_visu_b["top5_fontes"] = df_visu_b["top5_fontes"].apply(lambda x: " | ".join(x))
    df_visu_b = df_visu_b.sort_values(["query_id","variante","retrieval_mode","rerank_mode"]).reset_index(drop=True)
    display(
        df_visu_b.style
            .apply(colorir_acertos, axis=1)
            .format({"recall@5":"{:.2f}","precision@5":"{:.2f}","mrr":"{:.2f}","latency_ms":"{:.0f}"})
            .set_properties(**{"font-size":"10px"})
            .set_table_styles([{"selector":"th","props":[("background-color","#37474f"),("color","white")]}])
            .set_caption("Caminho B — 100 buscas")
    ) if 'display' in dir() else None

---

## 🏆 Passo 23 — Caminho B: 6 Cenários de Ablação


In [ ]:
if not df_neural.empty:
    # Melhor variante de rewriting no caminho B (com BM25 puro)
    variantes_recall_b = (
        df_neural[(df_neural["variante"] != "original") &
                   (df_neural["retrieval_mode"] == "bm25") &
                   (df_neural["rerank_mode"] == "off")]
        .groupby("variante")["recall@5"].mean().round(3).sort_values(ascending=False)
    )
    print("📊 Recall@5 médio por variante (BM25 puro, caminho B):")
    print(variantes_recall_b.to_string())
    melhor_variante_b = variantes_recall_b.idxmax() if not variantes_recall_b.empty else "paraphrase_1"
    print(f"\\n📌 Melhor variante (caminho B): '{melhor_variante_b}'")

    df_neural = aplicar_cenarios(df_neural, melhor_variante_b)
    resumo_b = resumo_cenarios(df_neural)
    print()
    print("📋 6 CENÁRIOS DE ABLAÇÃO — CAMINHO B:")
    print("=" * 100)
    print(resumo_b[["cenario_nome","recall_5_medio","precision_5_medio","mrr_medio","latency_medio",
                     "Δ_recall","Δ_precision","Δ_mrr","Δ_latency"]].to_string(index=False))

    display(
        resumo_b[["cenario_nome","recall_5_medio","precision_5_medio","mrr_medio","latency_medio",
                   "Δ_recall","Δ_precision","Δ_mrr","Δ_latency"]]
            .style
            .map(cor_pos, subset=["Δ_recall","Δ_precision","Δ_mrr"])
            .map(cor_neg, subset=["Δ_latency"])
            .format({"recall_5_medio":"{:.2f}","precision_5_medio":"{:.2f}","mrr_medio":"{:.2f}",
                     "Δ_recall":"{:+.2f}","Δ_precision":"{:+.2f}","Δ_mrr":"{:+.2f}",
                     "latency_medio":"{:.0f}","Δ_latency":"{:+.0f}"})
            .set_caption(f"Caminho B — Ablação dos 6 cenários (melhor rewriting: {melhor_variante_b})")
    ) if 'display' in dir() else None

---

## 📈 Passo 24 — Caminho B: Barplot de Ablação


In [ ]:
if not df_neural.empty and not resumo_b.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    metricas = [("recall_5_medio","Recall@5"), ("precision_5_medio","Precision@5"), ("mrr_medio","MRR")]
    labels6_b = [n.split(". ",1)[1][:18] for n in resumo_b["cenario_nome"]]
    cores6_b  = ["#90a4ae", "#8e24aa", "#039be5", "#7cb342", "#ef6c00", "#bf360c"]
    x = np.arange(len(labels6_b))

    for ax, (col, titulo) in zip(axes, metricas):
        vals = resumo_b[col].values
        ax.bar(x, vals, color=cores6_b, edgecolor="black", width=0.7)
        ax.set_xticks(x); ax.set_xticklabels(labels6_b, rotation=25, ha="right", fontsize=8)
        ax.set_title(f"{titulo} — Caminho B", fontweight="bold")
        ax.set_ylim(0, max(vals.max()*1.25, 0.05))
        ax.axhline(vals[0], color="#90a4ae", linestyle="--", alpha=0.6, label=f"baseline ({vals[0]:.2f})")
        for i,v in enumerate(vals):
            ax.text(i, v+0.005, f"{v:.2f}", ha="center", fontweight="bold", fontsize=8)
        ax.legend(loc="upper left", fontsize=7); ax.grid(axis="y", alpha=0.3)

    plt.suptitle("Caminho B — Ablação por cenário (OpenSearch ml-commons completo)",
                 fontsize=11, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.savefig("/tmp/lab2_v6_ablacao_B.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("✅ Gráfico B salvo: /tmp/lab2_v6_ablacao_B.png")

---

# 🆚 COMPARAÇÃO A vs B — 6 Cenários × 2 Caminhos

Junta os 2 resumos numa tabela mestre. Para cada cenário, mostramos as métricas
dos 2 caminhos e a diferença `Δ (B - A)`.


In [ ]:
if df_neural.empty or resumo_b.empty:
    print("ℹ️  Caminho B vazio — comparação A vs B não disponível")
    df_compare = pd.DataFrame()
else:
    # Merge dos 2 resumos
    df_compare = resumo_a[["cenario","cenario_nome","recall_5_medio","precision_5_medio","mrr_medio","latency_medio"]].copy()
    df_compare.columns = ["cenario","cenario_nome","A_recall","A_precision","A_mrr","A_latency"]
    df_compare = df_compare.merge(
        resumo_b[["cenario","recall_5_medio","precision_5_medio","mrr_medio","latency_medio"]]
            .rename(columns={"recall_5_medio":"B_recall","precision_5_medio":"B_precision",
                              "mrr_medio":"B_mrr","latency_medio":"B_latency"}),
        on="cenario", how="outer"
    ).sort_values("cenario")

    df_compare["Δ_recall (B-A)"]    = (df_compare["B_recall"]    - df_compare["A_recall"]).round(3)
    df_compare["Δ_mrr (B-A)"]       = (df_compare["B_mrr"]       - df_compare["A_mrr"]).round(3)
    df_compare["Δ_precision (B-A)"] = (df_compare["B_precision"] - df_compare["A_precision"]).round(3)
    df_compare["Δ_latency (B-A)"]   = (df_compare["B_latency"]   - df_compare["A_latency"]).round(0)

    print("📋 COMPARAÇÃO A vs B — 6 cenários × 2 caminhos")
    print("=" * 110)
    print(df_compare.to_string(index=False))

    display(
        df_compare[["cenario_nome","A_recall","B_recall","Δ_recall (B-A)",
                     "A_mrr","B_mrr","Δ_mrr (B-A)",
                     "A_latency","B_latency","Δ_latency (B-A)"]]
            .style
            .map(cor_pos, subset=["Δ_recall (B-A)","Δ_mrr (B-A)"])
            .map(cor_neg, subset=["Δ_latency (B-A)"])
            .format({"A_recall":"{:.2f}","B_recall":"{:.2f}","A_mrr":"{:.2f}","B_mrr":"{:.2f}",
                     "Δ_recall (B-A)":"{:+.2f}","Δ_mrr (B-A)":"{:+.2f}",
                     "A_latency":"{:.0f}","B_latency":"{:.0f}","Δ_latency (B-A)":"{:+.0f}"})
            .set_caption("A vs B: verde quando B vence em recall/MRR ou é mais rápido")
    ) if 'display' in dir() else None

---

## 🔥 Passo 26 — Heatmap: Cenário × Caminho (Recall@5)


In [ ]:
if not df_compare.empty:
    heatmap = df_compare[["A_recall","B_recall"]].copy()
    heatmap.index = [n[:25] for n in df_compare["cenario_nome"]]

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(heatmap.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(2)); ax.set_xticklabels(["Caminho A", "Caminho B"], fontsize=10, fontweight="bold")
    ax.set_yticks(range(len(heatmap.index)))
    ax.set_yticklabels(heatmap.index, fontsize=9)
    for i in range(len(heatmap.index)):
        for j in range(2):
            v = heatmap.values[i,j]
            cor_txt = "white" if v > 0.55 or v < 0.25 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color=cor_txt, fontsize=10, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Recall@5")
    ax.set_title("Heatmap Recall@5 — Cenário × Caminho", fontweight="bold")
    plt.tight_layout()
    plt.savefig("/tmp/lab2_v6_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("✅ Heatmap salvo: /tmp/lab2_v6_heatmap.png")

---

## 📈 Passo 27 — Barplot Triplo: Cenários A vs B


In [ ]:
if not df_compare.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metricas = [("recall","Recall@5","%.2f"),
                 ("mrr",   "MRR",       "%.2f"),
                 ("latency","Latência (ms)","%.0f")]
    labels = [n.split(". ",1)[1][:16] for n in df_compare["cenario_nome"]]
    x = np.arange(len(labels))
    w = 0.4

    for ax, (m, titulo, fmt) in zip(axes, metricas):
        vals_a = df_compare[f"A_{m}"].values
        vals_b = df_compare[f"B_{m}"].values
        ax.bar(x - w/2, vals_a, w, label="A (client)", color="#1976d2", edgecolor="black")
        ax.bar(x + w/2, vals_b, w, label="B (neural)", color="#f57c00", edgecolor="black")
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
        ax.set_title(titulo, fontweight="bold"); ax.set_ylabel(m)
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
        for i in range(len(labels)):
            ax.text(i-w/2, vals_a[i], fmt % vals_a[i], ha="center", va="bottom", fontsize=7)
            ax.text(i+w/2, vals_b[i], fmt % vals_b[i], ha="center", va="bottom", fontsize=7)

    plt.suptitle("Comparação A vs B em 6 cenários — Recall, MRR e Latência",
                 fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("/tmp/lab2_v6_a_vs_b_triplo.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("✅ Barplot triplo salvo: /tmp/lab2_v6_a_vs_b_triplo.png")

---

## 🔬 Passo 28 — Drill-down: Query com Maior Divergência A vs B


In [ ]:
if df_neural.empty:
    print("ℹ️  Sem dados B")
else:
    # Query×cenário "Advanced RAG completo" (5) — maior diferença A vs B
    deltas = []
    for qid in df_completo["query_id"].unique():
        a_row = df_completo[(df_completo["query_id"]==qid) & (df_completo["cenario"]==5)]
        b_row = df_neural[(df_neural["query_id"]==qid) & (df_neural["cenario"]==5)]
        if len(a_row) and len(b_row):
            ra, rb = a_row.iloc[0]["recall@5"], b_row.iloc[0]["recall@5"]
            deltas.append({"query_id": qid, "A_recall": ra, "B_recall": rb,
                           "delta": abs(ra-rb),
                           "A_ids": a_row.iloc[0]["top5_ids"], "B_ids": b_row.iloc[0]["top5_ids"],
                           "A_fontes": a_row.iloc[0]["top5_fontes"], "B_fontes": b_row.iloc[0]["top5_fontes"]})
    if deltas:
        df_d = pd.DataFrame(deltas).sort_values("delta", ascending=False)
        top = df_d.iloc[0]
        gt = ground_truth[top["query_id"]]
        q_obj = next(q for q in resultados_rewriting if q["id"]==top["query_id"])

        print(f"🔬 QUERY COM MAIOR DIVERGÊNCIA (cenário 5 — Advanced RAG completo): {top['query_id']}")
        print(f"   Pergunta: \"{q_obj['original']}\"")
        print(f"   A_recall={top['A_recall']:.2f}  |  B_recall={top['B_recall']:.2f}  |  Δ={top['delta']:.2f}")
        print(f"   Ground truth ({len(gt)} docs): {sorted(gt)}")
        print("=" * 75)
        print(f"\\n🟦 Caminho A (Advanced RAG via Ollama):")
        for i,(d,f) in enumerate(zip(top["A_ids"], top["A_fontes"]), 1):
            so_a = "🅰️" if d not in top["B_ids"] else "  "
            print(f"   {i}. {'✅' if d in gt else '❌'} {so_a} [{d}] {f[:55]}")
        print(f"\\n🟩 Caminho B (Advanced RAG via OpenSearch ml-commons):")
        for i,(d,f) in enumerate(zip(top["B_ids"], top["B_fontes"]), 1):
            so_b = "🅱️" if d not in top["A_ids"] else "  "
            print(f"   {i}. {'✅' if d in gt else '❌'} {so_b} [{d}] {f[:55]}")
        print(f"\\n💡 🅰️ = só A trouxe | 🅱️ = só B trouxe | ✅ = relevante")

---

## ⏱️ Passo 29 — Análise Dedicada de Latência


In [ ]:
if df_neural.empty:
    print("ℹ️  Sem dados B")
else:
    print("⏱  ANÁLISE DE LATÊNCIA — A vs B")
    print("=" * 75)
    print()
    stats_a = df_completo["latency_ms"].describe()
    stats_b = df_neural["latency_ms"].describe()
    print(f"📊 ESTATÍSTICAS (todas as buscas):")
    print(f"   {'Métrica':<10} {'A (client)':<15} {'B (neural)':<15} {'Δ B-A':<10}")
    for stat in ["mean", "50%", "min", "max"]:
        a, b = stats_a[stat], stats_b[stat]
        delta = b - a
        sinal = "↓" if delta < 0 else "↑"
        print(f"   {stat:<10} {a:>10.0f} ms  {b:>10.0f} ms  {sinal}{abs(delta):>+6.0f}")
    print()
    # Latência por cenário
    lat_cen_a = df_completo[df_completo["cenario"].notna()].groupby("cenario")["latency_ms"].mean().round(0)
    lat_cen_b = df_neural[df_neural["cenario"].notna()].groupby("cenario")["latency_ms"].mean().round(0)
    print(f"📊 LATÊNCIA MÉDIA POR CENÁRIO:")
    for cen in range(6):
        nome = CENARIOS_NOMES[cen][:35]
        la = lat_cen_a.get(cen, 0)
        lb = lat_cen_b.get(cen, 0)
        print(f"   {nome:<35} A: {la:>6.0f} ms  |  B: {lb:>6.0f} ms")
    print()

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.hist(df_completo["latency_ms"], bins=30, alpha=0.6, label="A (client)", color="#1976d2", edgecolor="black")
    ax.hist(df_neural["latency_ms"],   bins=30, alpha=0.6, label="B (neural)", color="#f57c00", edgecolor="black")
    ax.axvline(stats_a["mean"], color="#1976d2", linestyle="--", linewidth=2, label=f"média A ({stats_a['mean']:.0f}ms)")
    ax.axvline(stats_b["mean"], color="#f57c00", linestyle="--", linewidth=2, label=f"média B ({stats_b['mean']:.0f}ms)")
    ax.set_xlabel("Latência (ms)"); ax.set_ylabel("Frequência")
    ax.set_title("Distribuição de latência: Caminho A vs B (200 buscas)", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/lab2_v6_latency_hist.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("✅ Histograma salvo: /tmp/lab2_v6_latency_hist.png")